# 04. Гибридные модели: Two-Tower + GCMC

Ноутбук с экспериментами по ансамблированию двубашенной модели и графовой модели.

Логика экспериментов:
- отдельно обучается Two-Tower модель;
- отдельно обучается Support-GCMC;
- затем строится взвешенный ансамбль;
- проверяются варианты с замороженными компонентами и совместным дообучением.

Идея гибридного подхода в том, что двубашенная модель использует историю пользователя и признаки фильмов, а GCMC дополнительно учитывает структуру user-item графа с явными рейтингами.


In [ ]:
from __future__ import annotations


In [ ]:
try:
    import comet_ml  # noqa: F401
except Exception:
    try:
        get_ipython().run_line_magic("pip", "install -q comet_ml")
    except Exception as exc:
        print(f"[warning] comet_ml install skipped: {exc}")


In [ ]:
import os

# Для логирования в Comet задайте ключ вне ноутбука, например:
# export COMET_API_KEY="..."
# На GitHub ключи и токены не хранятся.
os.environ.setdefault("COMET_PROJECT_NAME", "explicit-movie-ranking")


In [ ]:
import json
import math
import os
import random
import time
from collections import defaultdict
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from comet_ml import Experiment
    COMET_AVAILABLE = True
except Exception:
    Experiment = None
    COMET_AVAILABLE = False


In [ ]:
@dataclass
class PathsConfig:
    dataset_dir: str = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
    output_dir: str = "/kaggle/working/explicit_rating_baselines"


@dataclass
class FeatureConfig:
    use_item_features: bool = True
    item_feature_mode: str = "add"   # none | add | project_only | concat_mlp | init_only
    scaler_kind: str = "standard"
    fit_scaler_on: str = "warm"      # warm | all
    feature_dropout: float = 0.10
    feature_hidden_dim: int = 256
    init_only_svd_n_iter: int = 7
    init_only_random_state: int = 42


@dataclass
class TrainConfig:
    seed: int = 42
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    epochs: int = 10
    lr: float = 3e-4
    weight_decay: float = 1e-5
    batch_size: int = 1024
    num_workers: int = 4 if (os.cpu_count() or 0) >= 4 else 2
    prefetch_factor: int = 2
    persistent_workers: bool = True
    patience: int = 3
    grad_clip_norm: Optional[float] = 5.0
    log_every_n_steps: int = 100
    verbose: bool = True
    positive_threshold: float = 4.0
    train_rating_weighting: str = "linear_norm"   # none | binary | raw | linear_norm | centered
    max_history: int = 50
    min_history_for_sequence: int = 1
    neg_samples: int = 1
    relation_edge_batch_size: int = 131072
    graph_loss_batches_per_epoch_log: int = 10
    use_amp: bool = True
    tf32: bool = True
    use_multi_gpu: bool = True
    compile_model: bool = False
    compile_mode: str = "default"


@dataclass
class EvalConfig:
    split_name: str = "val"  # kept for backward compatibility
    ks: Tuple[int, ...] = (10, 20, 50)
    metric_gain_types: Tuple[str, ...] = ("binary_ge_4", "centered_3", "power")
    binary_metric_names: Tuple[str, ...] = ("precision", "recall", "hitrate", "mrr", "map", "ndcg")
    graded_metric_names: Tuple[str, ...] = ("ndcg",)
    positive_threshold: float = 4.0
    neutral_rating: float = 3.0
    power_beta: float = 0.2
    power_gamma: float = 2.0
    exclude_seen_items: bool = True
    user_batch_size: int = 256
    item_batch_size: Optional[int] = None
    skip_users_with_no_relevant_targets: bool = True
    compute_loss: bool = True
    loss_batch_size: int = 4096
    loss_max_rows: Optional[int] = None
    score_mode: str = "default"  # default | expected_rating | prob_positive
    main_metric_name: str = "ndcg"
    main_metric_gain_type: str = "centered_3"
    main_metric_k: int = 10
    eval_scenarios: Tuple[str, ...] = ("warm", "cold_user", "cold_item", "both_cold")
    evaluate_cold_on_val: bool = True
    evaluate_cold_on_test: bool = True
    cold_user_use_support_all: bool = True


@dataclass
class CometConfig:
    enabled: bool = True
    project_name: str = "explicit-movie-ranking"
    workspace: Optional[str] = None
    experiment_name: Optional[str] = None
    tags: List[str] = field(default_factory=lambda: ["warm", "baseline"])
    log_code: bool = False


@dataclass
class GraphConfig:
    num_layers: int = 2
    embedding_dim: int = 128
    hidden_dim: int = 128
    dropout: float = 0.10
    relation_aggregation: str = "sum"   # sum | mean
    relation_apply_time_decay: bool = False
    time_weight_fn: str = "exp"
    time_weight_gamma: float = 1e-8
    rating_weight_fn: str = "linear_norm"
    relation_score_mode: str = "expected_rating"  # expected_rating | prob_positive
    aux_bpr_weight: float = 0.0
    aux_bpr_positive_threshold: float = 4.0
    aux_bpr_neg_samples: int = 1
    support_history_pooling: str = "weighted_mean"   # mean | weighted_mean | attention
    support_use_rating: bool = True
    support_use_time: bool = True
    support_time_weight_fn: str = "exp"
    support_time_weight_gamma: float = 1e-8
    support_alignment_weight: float = 0.2
    support_alignment_loss: str = "mse"   # mse | cosine
    support_user_batch_size: int = 4096


@dataclass
class TwoTowerConfig:
    embedding_dim: int = 128
    hidden_dim: int = 256
    dropout: float = 0.10
    use_user_id_embedding: bool = True
    # kept for backward compatibility with older notebook runs / saved configs
    use_history: bool = True
    history_pooling: str = "weighted_mean"
    use_rating_in_history: bool = True
    use_time_in_history: bool = True
    time_weight_fn: str = "exp"
    time_weight_gamma: float = 1e-8
    rating_weight_fn: str = "linear_norm"
    sasrec_num_layers: int = 2
    sasrec_num_heads: int = 4
    sasrec_ffn_dim: int = 256
    sasrec_attention_dropout: float = 0.10
    sasrec_max_seq_len: int = 512
    sasrec_use_causal_mask: bool = True


@dataclass
class MFConfig:
    embedding_dim: int = 128
    dropout: float = 0.0


@dataclass
class ModelConfig:
    model_type: str = "mf"   # mf | two_tower | relation_gcmc | support_gcmc
    mf: MFConfig = field(default_factory=MFConfig)
    two_tower: TwoTowerConfig = field(default_factory=TwoTowerConfig)
    graph: GraphConfig = field(default_factory=GraphConfig)


@dataclass
class ExperimentConfig:
    paths: PathsConfig = field(default_factory=PathsConfig)
    features: FeatureConfig = field(default_factory=FeatureConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)
    comet: CometConfig = field(default_factory=CometConfig)
    model: ModelConfig = field(default_factory=ModelConfig)


cfg = ExperimentConfig()
cfg.model.model_type = "mf"
# cfg.model.model_type = "two_tower"
# cfg.model.model_type = "relation_gcmc"
# cfg.model.model_type = "support_gcmc"

# Recommended first sweeps:
# cfg.train.batch_size = 512
# cfg.model.two_tower.embedding_dim = 192
# cfg.train.epochs = 10
# cfg.train.patience = 4
cfg


In [ ]:
def dc_to_dict(obj):
    if is_dataclass(obj):
        return {k: dc_to_dict(v) for k, v in asdict(obj).items()}
    if isinstance(obj, dict):
        return {k: dc_to_dict(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [dc_to_dict(v) for v in obj]
    return obj


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def configure_runtime(cfg: ExperimentConfig) -> None:
    if torch.cuda.is_available():
        if cfg.train.tf32:
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True


def use_amp(cfg: ExperimentConfig, device: torch.device) -> bool:
    return bool(cfg.train.use_amp and device.type == "cuda")


def get_autocast_context(cfg: ExperimentConfig, device: torch.device):
    if use_amp(cfg, device):
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler(cfg: ExperimentConfig, device: torch.device):
    return torch.cuda.amp.GradScaler(enabled=use_amp(cfg, device))


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def compute_grad_norm(parameters: Iterable[torch.nn.Parameter]) -> float:
    total_sq = 0.0
    for p in parameters:
        if p.grad is None:
            continue
        grad = p.grad.detach()
        total_sq += float(torch.sum(grad * grad).item())
    return float(total_sq ** 0.5)


def get_gpu_info() -> Dict[str, object]:
    if not torch.cuda.is_available():
        return {"num_gpus": 0, "gpu_names": []}
    return {
        "num_gpus": torch.cuda.device_count(),
        "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    }


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(data: dict, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path: str | Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def rating_weight(rating: np.ndarray | torch.Tensor, kind: str, r_min: float = 0.5, r_max: float = 5.0, neutral: float = 3.0):
    if isinstance(rating, torch.Tensor):
        x = rating.float()
        if kind == "none":
            return torch.ones_like(x)
        if kind == "binary":
            return (x >= 4.0).float()
        if kind == "raw":
            return x
        if kind == "linear_norm":
            return torch.clamp((x - r_min) / max(r_max - r_min, 1e-8), min=0.0)
        if kind == "centered":
            return torch.clamp(x - neutral, min=0.0)
        raise ValueError(f"Unknown rating weight kind: {kind}")
    x = np.asarray(rating, dtype=np.float32)
    if kind == "none":
        return np.ones_like(x)
    if kind == "binary":
        return (x >= 4.0).astype(np.float32)
    if kind == "raw":
        return x
    if kind == "linear_norm":
        return np.clip((x - r_min) / max(r_max - r_min, 1e-8), a_min=0.0, a_max=None).astype(np.float32)
    if kind == "centered":
        return np.clip(x - neutral, a_min=0.0, a_max=None).astype(np.float32)
    raise ValueError(f"Unknown rating weight kind: {kind}")


def time_weight(delta_seconds: np.ndarray | torch.Tensor, kind: str, gamma: float = 1e-8):
    if isinstance(delta_seconds, torch.Tensor):
        x = delta_seconds.float()
        if kind == "none":
            return torch.ones_like(x)
        if kind == "exp":
            return torch.exp(-gamma * x)
        if kind == "inverse":
            return 1.0 / (1.0 + gamma * x)
        if kind == "sqrt_inverse":
            return 1.0 / torch.sqrt(1.0 + gamma * x)
        raise ValueError(f"Unknown time weight kind: {kind}")
    x = np.asarray(delta_seconds, dtype=np.float32)
    if kind == "none":
        return np.ones_like(x)
    if kind == "exp":
        return np.exp(-gamma * x).astype(np.float32)
    if kind == "inverse":
        return (1.0 / (1.0 + gamma * x)).astype(np.float32)
    if kind == "sqrt_inverse":
        return (1.0 / np.sqrt(1.0 + gamma * x)).astype(np.float32)
    raise ValueError(f"Unknown time weight kind: {kind}")


def gain_from_rating(
    ratings: np.ndarray,
    gain_type: str,
    positive_threshold: float = 4.0,
    neutral_rating: float = 3.0,
    power_beta: float = 0.2,
    power_gamma: float = 2.0,
) -> np.ndarray:
    ratings = np.asarray(ratings, dtype=np.float32)
    if gain_type == "binary_ge_4":
        return (ratings >= positive_threshold).astype(np.float32)
    if gain_type == "centered_3":
        return np.clip(ratings - neutral_rating, a_min=0.0, a_max=None).astype(np.float32)
    if gain_type == "power":
        x = np.clip((ratings - 1.0) / 4.0, a_min=0.0, a_max=1.0)
        return (power_beta + (1.0 - power_beta) * np.power(x, power_gamma)).astype(np.float32)
    raise ValueError(f"Unknown gain_type={gain_type}")


def get_metric_gain_types(cfg_eval: EvalConfig) -> Tuple[str, ...]:
    gain_types = tuple(dict.fromkeys(cfg_eval.metric_gain_types or (cfg_eval.main_metric_gain_type,)))
    if not gain_types:
        gain_types = (cfg_eval.main_metric_gain_type,)
    return gain_types


def gain_type_metric_family(gain_type: str) -> str:
    if gain_type == "binary_ge_4":
        return "binary"
    if gain_type in {"centered_3", "power"}:
        return "graded"
    raise ValueError(f"Unknown gain_type={gain_type}")


In [ ]:
@dataclass
class PreparedData:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame
    eval_frames: Dict[str, pd.DataFrame]
    item_features: torch.Tensor
    item_feature_cols: List[str]
    num_users: int
    num_items: int
    num_warm_items: int
    warm_user2idx: Dict[str, int]
    warm_item2idx: Dict[str, int]
    all_item2idx: Dict[str, int]
    cold_user2idx: Dict[str, int]
    cold_item2idx: Dict[str, int]
    trainable_item_mask: torch.Tensor
    user_histories: Dict[int, List[Tuple[int, float, int]]]
    eval_histories: Dict[str, Dict[int, List[Tuple[int, float, int]]]]
    seen_items: Dict[int, set]
    eval_seen_items: Dict[str, Dict[int, set]]
    unique_ratings: List[float]
    rating2class: Dict[float, int]
    class2rating: Dict[int, float]
    output_dir: Path
    split_config: dict


REQUIRED_FILES = [
    "train_warm_interactions.parquet",
    "warm_val_interactions.parquet",
    "warm_test_interactions.parquet",
    "warm_user2idx.json",
    "warm_item2idx.json",
    "feature_cols.json",
]


def auto_find_dataset_dir(required_files: Sequence[str], preferred_path: str) -> Path:
    preferred = Path(preferred_path)
    if preferred.exists() and all((preferred / fname).exists() for fname in required_files):
        return preferred
    search_roots = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob(required_files[0]):
            candidate = path.parent
            if all((candidate / fname).exists() for fname in required_files):
                return candidate
    raise FileNotFoundError(
        f"Could not find dataset directory containing: {required_files}. Preferred path: {preferred_path}"
    )


def load_interactions(path: Path, user2idx: dict, item2idx: dict) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "user_idx" not in df.columns:
        df["user_idx"] = df["userId"].astype(str).map(user2idx)
    if "item_idx" not in df.columns:
        df["item_idx"] = df["movieId"].astype(str).map(item2idx)
    df = df.dropna(subset=["user_idx", "item_idx"]).copy()
    df["user_idx"] = df["user_idx"].astype(int)
    df["item_idx"] = df["item_idx"].astype(int)
    df["rating"] = df["rating"].astype(float)
    if "timestamp" in df.columns:
        df["timestamp"] = df["timestamp"].astype(np.int64)
    else:
        df["timestamp"] = 0
    return df


def prepare_item_feature_matrix(
    dataset_dir: Path,
    feature_cfg: FeatureConfig,
    all_item2idx: dict,
    warm_item_keys: Sequence[str],
    output_dir: Path,
) -> Tuple[torch.Tensor, List[str]]:
    feature_cols = load_json(dataset_dir / "feature_cols.json")
    feat_path = dataset_dir / "item_features_all.parquet"
    if not feat_path.exists():
        feat_path = dataset_dir / "item_features_warm.parquet"
    feat_df = pd.read_parquet(feat_path)
    feat_df["movieId"] = feat_df["movieId"].astype(str)
    feat_df = feat_df[feat_df["movieId"].isin(set(all_item2idx.keys()))].copy()
    feat_df["item_idx"] = feat_df["movieId"].map(all_item2idx).astype(int)
    feat_df = feat_df.sort_values("item_idx")

    for col in feature_cols:
        if col not in feat_df.columns:
            feat_df[col] = 0.0

    aligned = pd.DataFrame({"item_idx": np.arange(len(all_item2idx), dtype=np.int64)})
    aligned = aligned.merge(feat_df[["item_idx", *feature_cols]], on="item_idx", how="left")
    aligned[feature_cols] = aligned[feature_cols].fillna(0.0)

    x = aligned[feature_cols].astype(np.float32).to_numpy()
    if feature_cfg.scaler_kind == "standard":
        scaler = StandardScaler(with_mean=True, with_std=True)
        if feature_cfg.fit_scaler_on == "warm":
            warm_idx = sorted(int(all_item2idx[str(k)]) for k in warm_item_keys if str(k) in all_item2idx)
            fit_x = x[warm_idx] if warm_idx else x
        elif feature_cfg.fit_scaler_on == "all":
            fit_x = x
        else:
            raise ValueError(f"Unsupported fit_scaler_on={feature_cfg.fit_scaler_on}")
        scaler.fit(fit_x)
        x = scaler.transform(x).astype(np.float32)
        scaler_stats = {
            "mean": scaler.mean_.tolist(),
            "scale": scaler.scale_.tolist(),
            "feature_cols": feature_cols,
            "fit_scaler_on": feature_cfg.fit_scaler_on,
        }
        save_json(scaler_stats, output_dir / "feature_scaler_stats.json")
    elif feature_cfg.scaler_kind == "none":
        x = x.astype(np.float32)
    else:
        raise ValueError(f"Unsupported scaler_kind={feature_cfg.scaler_kind}")
    return torch.tensor(x, dtype=torch.float32), feature_cols


def build_user_histories(train_df: pd.DataFrame) -> Tuple[Dict[int, List[Tuple[int, float, int]]], Dict[int, set]]:
    histories: Dict[int, List[Tuple[int, float, int]]] = defaultdict(list)
    seen_items: Dict[int, set] = defaultdict(set)
    if train_df.empty:
        return {}, {}
    sort_df = train_df.sort_values(["user_idx", "timestamp", "item_idx"]).reset_index(drop=True)
    for row in sort_df.itertuples(index=False):
        histories[int(row.user_idx)].append((int(row.item_idx), float(row.rating), int(row.timestamp)))
        seen_items[int(row.user_idx)].add(int(row.item_idx))
    return dict(histories), dict(seen_items)


def load_optional_interactions(path: Path, user2idx: dict, item2idx: dict) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["userId", "movieId", "rating", "timestamp", "user_idx", "item_idx"])
    return load_interactions(path, user2idx, item2idx)


def load_prepared_data(cfg: ExperimentConfig) -> PreparedData:
    dataset_dir = auto_find_dataset_dir(REQUIRED_FILES, cfg.paths.dataset_dir)
    output_dir = ensure_dir(cfg.paths.output_dir)
    print(f"Loading dataset from: {dataset_dir}")

    warm_user2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "warm_user2idx.json").items()}
    warm_item2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "warm_item2idx.json").items()}
    if (dataset_dir / "all_item2idx.json").exists():
        all_item2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "all_item2idx.json").items()}
    else:
        all_item2idx = dict(warm_item2idx)
    cold_user2idx_local = {str(k): int(v) for k, v in load_json(dataset_dir / "cold_user2idx.json").items()} if (dataset_dir / "cold_user2idx.json").exists() else {}
    cold_item2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "cold_item2idx.json").items()} if (dataset_dir / "cold_item2idx.json").exists() else {}
    split_config = load_json(dataset_dir / "split_config.json") if (dataset_dir / "split_config.json").exists() else {}

    train_df = load_interactions(dataset_dir / "train_warm_interactions.parquet", warm_user2idx, all_item2idx)
    val_df = load_interactions(dataset_dir / "warm_val_interactions.parquet", warm_user2idx, all_item2idx)
    test_df = load_interactions(dataset_dir / "warm_test_interactions.parquet", warm_user2idx, all_item2idx)

    item_features, feature_cols = prepare_item_feature_matrix(
        dataset_dir,
        cfg.features,
        all_item2idx,
        warm_item_keys=list(warm_item2idx.keys()),
        output_dir=output_dir,
    )

    user_histories, seen_items = build_user_histories(train_df)
    warm_item_indices = sorted(int(all_item2idx[str(k)]) for k in warm_item2idx.keys() if str(k) in all_item2idx)
    trainable_item_mask = torch.zeros(len(all_item2idx), dtype=torch.float32)
    if warm_item_indices:
        trainable_item_mask[torch.tensor(warm_item_indices, dtype=torch.long)] = 1.0

    unique_ratings = sorted(float(x) for x in train_df["rating"].unique().tolist())
    rating2class = {r: i for i, r in enumerate(unique_ratings)}
    class2rating = {i: r for r, i in rating2class.items()}

    eval_frames: Dict[str, pd.DataFrame] = {
        "warm_val": val_df,
        "warm_test": test_df,
    }
    eval_histories: Dict[str, Dict[int, List[Tuple[int, float, int]]]] = {
        "warm_val": user_histories,
        "warm_test": user_histories,
    }
    eval_seen_items: Dict[str, Dict[int, set]] = {
        "warm_val": seen_items,
        "warm_test": seen_items,
    }

    if cold_user2idx_local:
        cold_user2idx = {raw_uid: len(warm_user2idx) + int(local_idx) for raw_uid, local_idx in cold_user2idx_local.items()}
        support_name = "cold_user_support_all.parquet" if cfg.eval.cold_user_use_support_all else "cold_user_support.parquet"
        cold_support_df = load_optional_interactions(dataset_dir / support_name, cold_user2idx, all_item2idx)
        cold_histories, cold_seen_items = build_user_histories(cold_support_df)

        cold_user_val_df = load_optional_interactions(dataset_dir / "cold_user_val_interactions.parquet", cold_user2idx, all_item2idx)
        cold_user_test_df = load_optional_interactions(dataset_dir / "cold_user_test_interactions.parquet", cold_user2idx, all_item2idx)
        both_cold_val_df = load_optional_interactions(dataset_dir / "both_cold_val_interactions.parquet", cold_user2idx, all_item2idx)
        both_cold_test_df = load_optional_interactions(dataset_dir / "both_cold_test_interactions.parquet", cold_user2idx, all_item2idx)

        eval_frames.update({
            "cold_user_val": cold_user_val_df,
            "cold_user_test": cold_user_test_df,
            "both_cold_val": both_cold_val_df,
            "both_cold_test": both_cold_test_df,
        })
        eval_histories.update({
            "cold_user_val": cold_histories,
            "cold_user_test": cold_histories,
            "both_cold_val": cold_histories,
            "both_cold_test": cold_histories,
        })
        eval_seen_items.update({
            "cold_user_val": cold_seen_items,
            "cold_user_test": cold_seen_items,
            "both_cold_val": cold_seen_items,
            "both_cold_test": cold_seen_items,
        })
    else:
        cold_user2idx = {}

    cold_item_val_df = load_optional_interactions(dataset_dir / "cold_item_val_interactions.parquet", warm_user2idx, all_item2idx)
    cold_item_test_df = load_optional_interactions(dataset_dir / "cold_item_test_interactions.parquet", warm_user2idx, all_item2idx)
    eval_frames.update({
        "cold_item_val": cold_item_val_df,
        "cold_item_test": cold_item_test_df,
    })
    eval_histories.update({
        "cold_item_val": user_histories,
        "cold_item_test": user_histories,
    })
    eval_seen_items.update({
        "cold_item_val": seen_items,
        "cold_item_test": seen_items,
    })

    return PreparedData(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        eval_frames=eval_frames,
        item_features=item_features,
        item_feature_cols=feature_cols,
        num_users=len(warm_user2idx),
        num_items=len(all_item2idx),
        num_warm_items=len(warm_item_indices),
        warm_user2idx=warm_user2idx,
        warm_item2idx=warm_item2idx,
        all_item2idx=all_item2idx,
        cold_user2idx=cold_user2idx,
        cold_item2idx=cold_item2idx,
        trainable_item_mask=trainable_item_mask,
        user_histories=user_histories,
        eval_histories=eval_histories,
        seen_items=seen_items,
        eval_seen_items=eval_seen_items,
        unique_ratings=unique_ratings,
        rating2class=rating2class,
        class2rating=class2rating,
        output_dir=output_dir,
        split_config=split_config,
    )


In [ ]:
class PairwiseTrainDataset(Dataset):
    def __init__(self, train_df: pd.DataFrame, seen_items: Dict[int, set], candidate_items: Sequence[int], positive_threshold: float):
        pos_df = train_df[train_df["rating"] >= positive_threshold].copy()
        self.users = pos_df["user_idx"].astype(np.int64).to_numpy()
        self.items = pos_df["item_idx"].astype(np.int64).to_numpy()
        self.ratings = pos_df["rating"].astype(np.float32).to_numpy()
        self.seen_items = seen_items
        self.candidate_items = np.asarray(list(candidate_items), dtype=np.int64)
        if self.candidate_items.size == 0:
            raise ValueError("candidate_items for negative sampling must be non-empty")

    def __len__(self) -> int:
        return len(self.users)

    def _sample_negative(self, user_idx: int) -> int:
        seen = self.seen_items.get(int(user_idx), set())
        while True:
            neg = int(self.candidate_items[random.randrange(len(self.candidate_items))])
            if neg not in seen:
                return neg

    def __getitem__(self, idx: int):
        u = int(self.users[idx])
        pos_i = int(self.items[idx])
        rating = float(self.ratings[idx])
        neg_i = self._sample_negative(u)
        return {
            "user_idx": u,
            "pos_item_idx": pos_i,
            "neg_item_idx": neg_i,
            "rating": rating,
        }


class SequencePairwiseDataset(Dataset):
    def __init__(
        self,
        train_df: pd.DataFrame,
        candidate_items: Sequence[int],
        positive_threshold: float,
        max_history: int,
        min_history: int,
    ):
        self.candidate_items = np.asarray(list(candidate_items), dtype=np.int32)
        if self.candidate_items.size == 0:
            raise ValueError("candidate_items for negative sampling must be non-empty")

        self.max_history = int(max_history)
        self.min_history = int(min_history)

        # Храним полные последовательности по пользователю, но только 1 раз
        self.user_items: Dict[int, np.ndarray] = {}
        self.user_ratings: Dict[int, np.ndarray] = {}
        self.user_timestamps: Dict[int, np.ndarray] = {}
        self.user_seen: Dict[int, set] = {}

        sample_users = []
        sample_pos = []

        df = train_df.sort_values(["user_idx", "timestamp", "item_idx"])

        for user_idx, g in tqdm(df.groupby("user_idx", sort=False), desc="Building sequence index"):
            items = g["item_idx"].to_numpy(dtype=np.int32, copy=True)
            ratings = g["rating"].to_numpy(dtype=np.float32, copy=True)
            timestamps = g["timestamp"].to_numpy(dtype=np.int64, copy=True)

            self.user_items[int(user_idx)] = items
            self.user_ratings[int(user_idx)] = ratings
            self.user_timestamps[int(user_idx)] = timestamps
            self.user_seen[int(user_idx)] = set(map(int, items.tolist()))

            # valid target positions: positive item и уже есть история достаточной длины
            pos_mask = ratings >= positive_threshold
            pos_idx = np.flatnonzero(pos_mask)
            pos_idx = pos_idx[pos_idx >= self.min_history]

            if len(pos_idx) > 0:
                sample_users.extend([int(user_idx)] * len(pos_idx))
                sample_pos.extend(pos_idx.tolist())

        self.sample_users = np.asarray(sample_users, dtype=np.int32)
        self.sample_pos = np.asarray(sample_pos, dtype=np.int32)

    def __len__(self) -> int:
        return len(self.sample_users)

    def _sample_negative(self, user_idx: int) -> int:
        seen = self.user_seen[user_idx]
        while True:
            neg = int(self.candidate_items[random.randrange(len(self.candidate_items))])
            if neg not in seen:
                return neg

    def __getitem__(self, idx: int):
        user_idx = int(self.sample_users[idx])
        pos = int(self.sample_pos[idx])

        items = self.user_items[user_idx]
        ratings = self.user_ratings[user_idx]
        timestamps = self.user_timestamps[user_idx]

        start = max(0, pos - self.max_history)

        hist_items = items[start:pos]
        hist_ratings = ratings[start:pos]
        hist_timestamps = timestamps[start:pos]

        pos_item_idx = int(items[pos])
        rating = float(ratings[pos])
        target_timestamp = int(timestamps[pos])

        neg_i = self._sample_negative(user_idx)

        return {
            "user_idx": user_idx,
            "hist_items": hist_items,
            "hist_ratings": hist_ratings,
            "hist_timestamps": hist_timestamps,
            "pos_item_idx": pos_item_idx,
            "neg_item_idx": neg_i,
            "rating": rating,
            "target_timestamp": target_timestamp,
        }



class ObservedEdgeDataset(Dataset):
    def __init__(self, train_df: pd.DataFrame, rating2class: Dict[float, int]):
        self.users = train_df["user_idx"].astype(np.int64).to_numpy()
        self.items = train_df["item_idx"].astype(np.int64).to_numpy()
        self.ratings = train_df["rating"].astype(np.float32).to_numpy()
        self.timestamps = train_df["timestamp"].astype(np.int64).to_numpy()
        self.classes = np.asarray([rating2class[float(r)] for r in self.ratings], dtype=np.int64)

    def __len__(self) -> int:
        return len(self.users)

    def __getitem__(self, idx: int):
        return {
            "user_idx": int(self.users[idx]),
            "item_idx": int(self.items[idx]),
            "rating": float(self.ratings[idx]),
            "timestamp": int(self.timestamps[idx]),
            "rating_class": int(self.classes[idx]),
        }


def sequence_collate_fn(batch: List[dict]):
    bsz = len(batch)
    max_len = max(len(x["hist_items"]) for x in batch)
    hist_items = torch.zeros((bsz, max_len), dtype=torch.long)
    hist_ratings = torch.zeros((bsz, max_len), dtype=torch.float32)
    hist_timestamps = torch.zeros((bsz, max_len), dtype=torch.long)
    hist_mask = torch.zeros((bsz, max_len), dtype=torch.bool)
    user_idx = torch.zeros(bsz, dtype=torch.long)
    pos_item_idx = torch.zeros(bsz, dtype=torch.long)
    neg_item_idx = torch.zeros(bsz, dtype=torch.long)
    rating = torch.zeros(bsz, dtype=torch.float32)
    target_timestamp = torch.zeros(bsz, dtype=torch.long)

    for i, sample in enumerate(batch):
        n = len(sample["hist_items"])
        user_idx[i] = sample["user_idx"]
        pos_item_idx[i] = sample["pos_item_idx"]
        neg_item_idx[i] = sample["neg_item_idx"]
        rating[i] = sample["rating"]
        target_timestamp[i] = sample["target_timestamp"]
        hist_items[i, :n] = torch.tensor(sample["hist_items"], dtype=torch.long)
        hist_ratings[i, :n] = torch.tensor(sample["hist_ratings"], dtype=torch.float32)
        hist_timestamps[i, :n] = torch.tensor(sample["hist_timestamps"], dtype=torch.long)
        hist_mask[i, :n] = True

    return {
        "user_idx": user_idx,
        "pos_item_idx": pos_item_idx,
        "neg_item_idx": neg_item_idx,
        "rating": rating,
        "target_timestamp": target_timestamp,
        "hist_items": hist_items,
        "hist_ratings": hist_ratings,
        "hist_timestamps": hist_timestamps,
        "hist_mask": hist_mask,
    }


In [ ]:
def build_eval_targets(eval_df: pd.DataFrame, cfg_eval: EvalConfig, gain_type: str) -> Dict[int, Dict[int, float]]:
    targets: Dict[int, Dict[int, float]] = defaultdict(dict)
    if eval_df.empty:
        return {}
    grouped = eval_df.groupby("user_idx")
    for user_idx, g in grouped:
        gains = gain_from_rating(
            g["rating"].to_numpy(),
            gain_type=gain_type,
            positive_threshold=cfg_eval.positive_threshold,
            neutral_rating=cfg_eval.neutral_rating,
            power_beta=cfg_eval.power_beta,
            power_gamma=cfg_eval.power_gamma,
        )
        items = g["item_idx"].astype(int).to_numpy()
        tgt = {int(i): float(gain) for i, gain in zip(items, gains) if gain > 0}
        if tgt or not cfg_eval.skip_users_with_no_relevant_targets:
            targets[int(user_idx)] = tgt
    return dict(targets)


def dcg_at_k(gains: Sequence[float], k: int) -> float:
    gains = np.asarray(gains[:k], dtype=np.float32)
    if gains.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))


def compute_ranking_metrics(
    ranked_items: np.ndarray,
    target_gain_dict: Dict[int, float],
    ks: Sequence[int],
    metric_names: Sequence[str],
) -> Dict[str, float]:
    metrics: Dict[str, float] = {}
    gains_ranked = np.asarray([target_gain_dict.get(int(i), 0.0) for i in ranked_items], dtype=np.float32)
    binary_ranked = (gains_ranked > 0).astype(np.float32)
    relevant_total = int(np.sum(np.asarray(list(target_gain_dict.values()), dtype=np.float32) > 0))
    ideal_gains = sorted(target_gain_dict.values(), reverse=True)

    for k in ks:
        topk_gains = gains_ranked[:k]
        topk_binary = binary_ranked[:k]
        hits = int(np.sum(topk_binary))

        if "hitrate" in metric_names:
            metrics[f"hitrate@{k}"] = float(hits > 0)
        if "precision" in metric_names:
            metrics[f"precision@{k}"] = float(hits / max(k, 1))
        if "recall" in metric_names:
            metrics[f"recall@{k}"] = float(hits / relevant_total) if relevant_total > 0 else 0.0
        if "ndcg" in metric_names:
            dcg = dcg_at_k(topk_gains, k)
            idcg = dcg_at_k(ideal_gains, min(k, len(ideal_gains)))
            metrics[f"ndcg@{k}"] = float(dcg / idcg) if idcg > 0 else 0.0
        if "mrr" in metric_names or "map" in metric_names:
            rel_positions = np.flatnonzero(topk_binary > 0)
            if "mrr" in metric_names:
                metrics[f"mrr@{k}"] = float(1.0 / (rel_positions[0] + 1)) if rel_positions.size > 0 else 0.0
            if "map" in metric_names:
                if rel_positions.size > 0:
                    prefix_hits = np.cumsum(topk_binary)
                    precisions = prefix_hits[rel_positions] / (rel_positions + 1)
                    ap_denom = min(relevant_total, k)
                    metrics[f"map@{k}"] = float(np.sum(precisions) / ap_denom) if ap_denom > 0 else 0.0
                else:
                    metrics[f"map@{k}"] = 0.0
    return metrics


def aggregate_metrics(metric_list: List[Dict[str, float]]) -> Dict[str, float]:
    if not metric_list:
        return {}
    keys = sorted(set().union(*metric_list))
    return {k: float(np.mean([row.get(k, 0.0) for row in metric_list])) for k in keys}


class ItemRepresentationMixin:
    def build_item_base(self, item_idx: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class FeatureAwareItemEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        feature_tensor: torch.Tensor,
        embedding_dim: int,
        feature_cfg: FeatureConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.embedding_dim = int(embedding_dim)
        self.feature_cfg = feature_cfg
        self.item_id_emb = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.item_id_emb.weight, std=0.02)

        self.register_buffer("item_features", feature_tensor)
        if trainable_item_mask is None:
            trainable_item_mask = torch.ones(num_items, dtype=torch.float32)
        self.register_buffer("trainable_item_mask", trainable_item_mask.float().view(-1, 1))
        feat_dim = int(feature_tensor.shape[1]) if feature_tensor is not None else 0
        self.use_features = feature_tensor is not None and feature_cfg.use_item_features and feat_dim > 0
        self.feature_dropout = nn.Dropout(feature_cfg.feature_dropout)

        if self.use_features:
            self.feature_proj = nn.Linear(feat_dim, embedding_dim)
            nn.init.xavier_uniform_(self.feature_proj.weight)
            nn.init.zeros_(self.feature_proj.bias)
            if feature_cfg.item_feature_mode == "concat_mlp":
                self.concat_mlp = nn.Sequential(
                    nn.Linear(embedding_dim * 2, feature_cfg.feature_hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(feature_cfg.feature_dropout),
                    nn.Linear(feature_cfg.feature_hidden_dim, embedding_dim),
                )
        else:
            self.feature_proj = None
            self.concat_mlp = None

    def forward(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_id_vec = self.item_id_emb(item_idx) * self.trainable_item_mask[item_idx]
        if not self.use_features:
            return item_id_vec
        feat_vec = self.feature_proj(self.feature_dropout(self.item_features[item_idx]))
        mode = self.feature_cfg.item_feature_mode
        if mode == "none":
            return item_id_vec
        if mode == "add":
            return item_id_vec + feat_vec
        if mode == "project_only":
            return feat_vec
        if mode == "concat_mlp":
            return self.concat_mlp(torch.cat([item_id_vec, feat_vec], dim=-1))
        raise ValueError(f"Unknown item_feature_mode={mode}")

    def all_item_vectors(self) -> torch.Tensor:
        idx = torch.arange(self.num_items, device=self.item_id_emb.weight.device)
        return self.forward(idx)


class MFModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        mf_cfg: MFConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, mf_cfg.embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.02)
        self.item_encoder = FeatureAwareItemEncoder(num_items, feature_tensor, mf_cfg.embedding_dim, feature_cfg, trainable_item_mask)
        self.dropout = nn.Dropout(mf_cfg.dropout)

    def score(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.dropout(self.user_emb(user_idx))
        v = self.dropout(self.item_encoder(item_idx))
        return (u * v).sum(dim=-1)

    def forward(self, user_idx: torch.Tensor, pos_item_idx: torch.Tensor, neg_item_idx: Optional[torch.Tensor] = None):
        pos_scores = self.score(user_idx, pos_item_idx)
        if neg_item_idx is None:
            return pos_scores
        neg_scores = self.score(user_idx, neg_item_idx)
        return pos_scores, neg_scores

    def score_all_items(self, user_idx: torch.Tensor) -> torch.Tensor:
        u = self.user_emb(user_idx)
        all_items = self.item_encoder.all_item_vectors()
        return u @ all_items.T


class AttentionPooling(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor, weights: Optional[torch.Tensor] = None) -> torch.Tensor:
        logits = self.proj(x).squeeze(-1)
        if weights is not None:
            logits = logits + torch.log(torch.clamp(weights, min=1e-8))
        logits = logits.masked_fill(~mask, -1e9)
        attn = torch.softmax(logits, dim=-1)
        return torch.bmm(attn.unsqueeze(1), x).squeeze(1)



class SASRecSequenceEncoder(nn.Module):
    def __init__(
        self,
        embedding_dim: int,
        max_seq_len: int,
        num_layers: int,
        num_heads: int,
        ffn_dim: int,
        dropout: float,
        attention_dropout: float,
        use_causal_mask: bool = True,
    ):
        super().__init__()
        if embedding_dim % num_heads != 0:
            raise ValueError("TwoTowerConfig.embedding_dim must be divisible by sasrec_num_heads")
        self.embedding_dim = int(embedding_dim)
        self.max_seq_len = int(max_seq_len)
        self.use_causal_mask = bool(use_causal_mask)

        self.position_emb = nn.Embedding(self.max_seq_len, self.embedding_dim)
        nn.init.normal_(self.position_emb.weight, std=0.02)
        self.empty_history_token = nn.Parameter(torch.zeros(self.embedding_dim))
        nn.init.normal_(self.empty_history_token, std=0.02)

        self.input_norm = nn.LayerNorm(self.embedding_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=attention_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(self.embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def _build_causal_mask(self, seq_len: int, device: torch.device) -> Optional[torch.Tensor]:
        if not self.use_causal_mask or seq_len <= 1:
            return None
        return torch.triu(torch.ones((seq_len, seq_len), dtype=torch.bool, device=device), diagonal=1)

    def forward(self, seq_embeddings: torch.Tensor, hist_mask: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = seq_embeddings.shape
        if seq_len > self.max_seq_len:
            seq_embeddings = seq_embeddings[:, -self.max_seq_len:, :]
            hist_mask = hist_mask[:, -self.max_seq_len:]
            seq_len = self.max_seq_len

        pos_idx = torch.arange(seq_len, device=seq_embeddings.device, dtype=torch.long).unsqueeze(0)
        x = seq_embeddings + self.position_emb(pos_idx)

        effective_mask = hist_mask.clone()
        has_history = effective_mask.any(dim=1)
        if (~has_history).any():
            empty_rows = (~has_history).nonzero(as_tuple=False).squeeze(-1)
            x = x.clone()
            x[empty_rows, 0, :] = self.empty_history_token
            effective_mask[empty_rows, 0] = True

        x = self.input_norm(x)
        x = self.dropout(x)
        x = self.encoder(
            x,
            mask=self._build_causal_mask(seq_len, x.device),
            src_key_padding_mask=~effective_mask,
        )
        x = self.output_norm(x)

        last_idx = effective_mask.long().sum(dim=1).clamp(min=1) - 1
        user_vec = x[torch.arange(batch_size, device=x.device), last_idx]
        return user_vec


class TwoTowerModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        model_cfg: TwoTowerConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.model_cfg = model_cfg
        self.embedding_dim = model_cfg.embedding_dim
        self.num_users = int(num_users)
        self.item_encoder = FeatureAwareItemEncoder(
            num_items,
            feature_tensor,
            model_cfg.embedding_dim,
            feature_cfg,
            trainable_item_mask,
        )
        self.use_user_id = model_cfg.use_user_id_embedding
        if self.use_user_id:
            self.user_id_emb = nn.Embedding(num_users, model_cfg.embedding_dim)
            nn.init.normal_(self.user_id_emb.weight, std=0.02)
        else:
            self.user_id_emb = None

        self.rating_proj = nn.Sequential(
            nn.Linear(1, model_cfg.embedding_dim),
            nn.GELU(),
            nn.Linear(model_cfg.embedding_dim, model_cfg.embedding_dim),
        )
        self.time_proj = nn.Sequential(
            nn.Linear(1, model_cfg.embedding_dim),
            nn.GELU(),
            nn.Linear(model_cfg.embedding_dim, model_cfg.embedding_dim),
        )
        self.sequence_encoder = SASRecSequenceEncoder(
            embedding_dim=model_cfg.embedding_dim,
            max_seq_len=model_cfg.sasrec_max_seq_len,
            num_layers=model_cfg.sasrec_num_layers,
            num_heads=model_cfg.sasrec_num_heads,
            ffn_dim=model_cfg.sasrec_ffn_dim,
            dropout=model_cfg.dropout,
            attention_dropout=model_cfg.sasrec_attention_dropout,
            use_causal_mask=model_cfg.sasrec_use_causal_mask,
        )
        self.user_mlp = nn.Sequential(
            nn.Linear(model_cfg.embedding_dim * (2 if self.use_user_id else 1), model_cfg.hidden_dim),
            nn.GELU(),
            nn.Dropout(model_cfg.dropout),
            nn.Linear(model_cfg.hidden_dim, model_cfg.embedding_dim),
        )
        # Item tower follows the residual feature-aware scheme:
        #   warm item: v_i = e_i + W x_i
        #   cold item: v_i = W x_i
        # implemented through FeatureAwareItemEncoder + trainable_item_mask
        self.dropout = nn.Dropout(model_cfg.dropout)

    def build_history_tokens(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_vecs = self.item_encoder(hist_items)

        if self.model_cfg.use_rating_in_history:
            seq_vecs = seq_vecs + self.rating_proj(hist_ratings.unsqueeze(-1))

        if self.model_cfg.use_time_in_history and target_timestamp is not None:
            delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0).float()
            time_input = torch.log1p(delta).unsqueeze(-1)
            seq_vecs = seq_vecs + self.time_proj(time_input)

        token_weights = hist_mask.float()
        if self.model_cfg.use_rating_in_history:
            token_weights = token_weights * rating_weight(hist_ratings, self.model_cfg.rating_weight_fn)
        if self.model_cfg.use_time_in_history and target_timestamp is not None:
            delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0).float()
            token_weights = token_weights * time_weight(
                delta,
                self.model_cfg.time_weight_fn,
                self.model_cfg.time_weight_gamma,
            )

        seq_vecs = seq_vecs * token_weights.unsqueeze(-1)
        return self.dropout(seq_vecs)

    def encode_history(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_vecs = self.build_history_tokens(
            hist_items=hist_items,
            hist_ratings=hist_ratings,
            hist_timestamps=hist_timestamps,
            hist_mask=hist_mask,
            target_timestamp=target_timestamp,
        )
        return self.sequence_encoder(seq_vecs, hist_mask)

    def encode_user(
        self,
        user_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        parts = [
            self.encode_history(
                hist_items=hist_items,
                hist_ratings=hist_ratings,
                hist_timestamps=hist_timestamps,
                hist_mask=hist_mask,
                target_timestamp=target_timestamp,
            )
        ]
        if self.use_user_id:
            valid_user = ((user_idx >= 0) & (user_idx < self.num_users)).float().unsqueeze(-1)
            safe_user_idx = torch.where(
                user_idx >= 0,
                torch.clamp(user_idx, max=self.num_users - 1),
                torch.zeros_like(user_idx),
            )
            parts.append(self.user_id_emb(safe_user_idx) * valid_user)
        user_vec = torch.cat(parts, dim=-1)
        return self.user_mlp(self.dropout(user_vec))

    def encode_item(self, item_idx: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.item_encoder(item_idx))

    def score_batch(self, batch: dict) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.forward(
            batch["user_idx"],
            batch["pos_item_idx"],
            batch["neg_item_idx"],
            batch["hist_items"],
            batch["hist_ratings"],
            batch["hist_timestamps"],
            batch["hist_mask"],
            batch["target_timestamp"],
        )

    def forward(
        self,
        user_idx: torch.Tensor,
        pos_item_idx: torch.Tensor,
        neg_item_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_vec = self.encode_user(
            user_idx,
            hist_items,
            hist_ratings,
            hist_timestamps,
            hist_mask,
            target_timestamp,
        )
        pos_vec = self.encode_item(pos_item_idx)
        neg_vec = self.encode_item(neg_item_idx)
        pos_scores = (user_vec * pos_vec).sum(dim=-1)
        neg_scores = (user_vec * neg_vec).sum(dim=-1)
        return pos_scores, neg_scores

    def score_all_items_for_histories(self, user_vecs: torch.Tensor) -> torch.Tensor:
        item_vecs = self.encode_item(torch.arange(self.item_encoder.num_items, device=user_vecs.device))
        return user_vecs @ item_vecs.T


class RelationGCMLayer(nn.Module):
    def __init__(self, dim_in: int, dim_out: int, num_relations: int, dropout: float):
        super().__init__()
        self.user_rel_linears = nn.ModuleList([nn.Linear(dim_in, dim_out, bias=False) for _ in range(num_relations)])
        self.item_rel_linears = nn.ModuleList([nn.Linear(dim_in, dim_out, bias=False) for _ in range(num_relations)])
        self.user_self = nn.Linear(dim_in, dim_out, bias=True)
        self.item_self = nn.Linear(dim_in, dim_out, bias=True)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        user_h: torch.Tensor,
        item_h: torch.Tensor,
        adjs_u_from_i: Sequence[torch.Tensor],
        adjs_i_from_u: Sequence[torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_msg = self.user_self(user_h)
        item_msg = self.item_self(item_h)
        for lin_u, lin_i, a_ui, a_iu in zip(self.user_rel_linears, self.item_rel_linears, adjs_u_from_i, adjs_i_from_u):
            user_msg = user_msg + lin_u(torch.sparse.mm(a_ui, item_h))
            item_msg = item_msg + lin_i(torch.sparse.mm(a_iu, user_h))
        user_out = F.relu(self.dropout(user_msg))
        item_out = F.relu(self.dropout(item_msg))
        return user_out, item_out


class BilinearRatingDecoder(nn.Module):
    def __init__(self, dim: int, num_relations: int):
        super().__init__()
        self.Q = nn.Parameter(torch.empty(num_relations, dim, dim))
        nn.init.xavier_uniform_(self.Q)

    def forward(self, user_vec: torch.Tensor, item_vec: torch.Tensor) -> torch.Tensor:
        # logits: [B, R]
        logits = torch.einsum("bd,rdh,bh->br", user_vec, self.Q, item_vec)
        return logits


class RelationGCMCModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        graph_cfg: GraphConfig,
        rating_values: List[float],
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.graph_cfg = graph_cfg
        self.rating_values = [float(x) for x in rating_values]
        self.num_relations = len(rating_values)
        self.user_emb = nn.Embedding(num_users, graph_cfg.embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.02)
        self.item_encoder = FeatureAwareItemEncoder(num_items, feature_tensor, graph_cfg.embedding_dim, feature_cfg, trainable_item_mask)
        self.layers = nn.ModuleList(
            [
                RelationGCMLayer(graph_cfg.embedding_dim, graph_cfg.embedding_dim, self.num_relations, graph_cfg.dropout)
                for _ in range(graph_cfg.num_layers)
            ]
        )
        self.decoder = BilinearRatingDecoder(graph_cfg.embedding_dim, self.num_relations)
        self._cached_graph = None

    def set_graph(self, graph_dict: dict) -> None:
        self._cached_graph = graph_dict

    def encode_full_graph(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if self._cached_graph is None:
            raise RuntimeError("Graph is not set. Call model.set_graph(graph_dict) first.")
        adjs_u_from_i = self._cached_graph["adjs_u_from_i"]
        adjs_i_from_u = self._cached_graph["adjs_i_from_u"]
        user_h = self.user_emb.weight
        item_h = self.item_encoder.all_item_vectors()
        user_states = [user_h]
        item_states = [item_h]
        for layer in self.layers:
            user_h, item_h = layer(user_h, item_h, adjs_u_from_i, adjs_i_from_u)
            user_states.append(user_h)
            item_states.append(item_h)
        user_z = torch.stack(user_states, dim=0).mean(dim=0)
        item_z = torch.stack(item_states, dim=0).mean(dim=0)
        return user_z, item_z

    def decode(self, user_vec: torch.Tensor, item_vec: torch.Tensor) -> torch.Tensor:
        return self.decoder(user_vec, item_vec)

    def _score_weights(self, device: torch.device, dtype: torch.dtype, positive_threshold: float = 4.0) -> torch.Tensor:
        relation_values = torch.tensor(self.rating_values, device=device, dtype=dtype)
        if self.graph_cfg.relation_score_mode == "prob_positive":
            return (relation_values >= positive_threshold).float().to(dtype=dtype)
        return relation_values

    def score_from_logits(self, logits: torch.Tensor, positive_threshold: float = 4.0) -> torch.Tensor:
        probs = torch.softmax(logits, dim=-1)
        weights = self._score_weights(logits.device, logits.dtype, positive_threshold)
        return probs @ weights

    def score_user_vectors_all_items(
        self,
        user_vecs: torch.Tensor,
        item_vecs: torch.Tensor,
        positive_threshold: float = 4.0,
    ) -> torch.Tensor:
        projected_users = torch.einsum("bd,rdh->brh", user_vecs, self.decoder.Q)
        logits = torch.einsum("brh,nh->bnr", projected_users, item_vecs)
        probs = torch.softmax(logits, dim=-1)
        weights = self._score_weights(user_vecs.device, user_vecs.dtype, positive_threshold)
        return torch.einsum("bnr,r->bn", probs, weights)

    def score_all_items(self, user_indices: torch.Tensor) -> torch.Tensor:
        user_z, item_z = self.encode_full_graph()
        return self.score_user_vectors_all_items(user_z[user_indices], item_z, positive_threshold=4.0)


class SupportAwareGCMCModel(RelationGCMCModel):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        graph_cfg: GraphConfig,
        rating_values: List[float],
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__(
            num_users=num_users,
            num_items=num_items,
            feature_tensor=feature_tensor,
            feature_cfg=feature_cfg,
            graph_cfg=graph_cfg,
            rating_values=rating_values,
            trainable_item_mask=trainable_item_mask,
        )
        self.relation_emb = nn.Embedding(self.num_relations, graph_cfg.embedding_dim)
        nn.init.normal_(self.relation_emb.weight, std=0.02)
        self.support_attention_pool = AttentionPooling(graph_cfg.embedding_dim)
        self.support_user_mlp = nn.Sequential(
            nn.Linear(graph_cfg.embedding_dim, graph_cfg.hidden_dim),
            nn.ReLU(),
            nn.Dropout(graph_cfg.dropout),
            nn.Linear(graph_cfg.hidden_dim, graph_cfg.embedding_dim),
        )

    def ratings_to_relation_ids(self, ratings: torch.Tensor) -> torch.Tensor:
        relation_values = torch.tensor(self.rating_values, device=ratings.device, dtype=ratings.dtype)
        rel_ids = torch.argmin(torch.abs(ratings.unsqueeze(-1) - relation_values.view(*([1] * ratings.dim()), -1)), dim=-1)
        return rel_ids.long()

    def encode_support_users(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
        item_vectors: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if item_vectors is None:
            _, item_vectors = self.encode_full_graph()
        hist_vecs = item_vectors[hist_items]
        rel_ids = self.ratings_to_relation_ids(hist_ratings)
        hist_vecs = hist_vecs + self.relation_emb(rel_ids)

        weights = hist_mask.float()
        if self.graph_cfg.support_use_rating:
            weights = weights * rating_weight(hist_ratings, self.graph_cfg.rating_weight_fn)
        if self.graph_cfg.support_use_time and target_timestamp is not None:
            delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0)
            weights = weights * time_weight(delta.float(), self.graph_cfg.support_time_weight_fn, self.graph_cfg.support_time_weight_gamma)

        pooling = self.graph_cfg.support_history_pooling
        if pooling == "attention":
            pooled = self.support_attention_pool(hist_vecs, hist_mask, weights)
        else:
            denom = hist_mask.float().sum(dim=1, keepdim=True).clamp(min=1.0) if pooling == "mean" else weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
            pooled = (hist_vecs * weights.unsqueeze(-1)).sum(dim=1) / denom
        pooled = pooled * hist_mask.any(dim=1, keepdim=True).float()
        return self.support_user_mlp(pooled)


In [ ]:

def make_sparse_coo(indices: np.ndarray, values: np.ndarray, size: Tuple[int, int], device: torch.device) -> torch.Tensor:
    idx = torch.tensor(indices, dtype=torch.long, device=device)
    vals = torch.tensor(values, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(idx, vals, size=size, device=device).coalesce()


def build_relation_graph(
    train_df: pd.DataFrame,
    num_users: int,
    num_items: int,
    rating_values: List[float],
    device: torch.device,
    apply_time_decay: bool = False,
    time_weight_fn_name: str = "exp",
    gamma: float = 1e-8,
) -> dict:
    latest_ts = int(train_df["timestamp"].max()) if len(train_df) else 0
    adjs_u_from_i: List[torch.Tensor] = []
    adjs_i_from_u: List[torch.Tensor] = []
    for rating in rating_values:
        rel_df = train_df[train_df["rating"] == rating]
        if len(rel_df) == 0:
            empty_ui = make_sparse_coo(np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32), (num_users, num_items), device)
            empty_iu = make_sparse_coo(np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32), (num_items, num_users), device)
            adjs_u_from_i.append(empty_ui)
            adjs_i_from_u.append(empty_iu)
            continue
        u = rel_df["user_idx"].astype(np.int64).to_numpy()
        i = rel_df["item_idx"].astype(np.int64).to_numpy()
        w = np.ones(len(rel_df), dtype=np.float32)
        if apply_time_decay:
            delta = (latest_ts - rel_df["timestamp"].astype(np.int64).to_numpy()).clip(min=0)
            w *= time_weight(delta, time_weight_fn_name, gamma)

        user_deg = np.bincount(u, weights=w, minlength=num_users).astype(np.float32)
        item_deg = np.bincount(i, weights=w, minlength=num_items).astype(np.float32)
        norm = w / np.sqrt(np.maximum(user_deg[u], 1e-8) * np.maximum(item_deg[i], 1e-8))

        ui_idx = np.vstack([u, i])
        iu_idx = np.vstack([i, u])
        a_ui = make_sparse_coo(ui_idx, norm, (num_users, num_items), device)
        a_iu = make_sparse_coo(iu_idx, norm, (num_items, num_users), device)
        adjs_u_from_i.append(a_ui)
        adjs_i_from_u.append(a_iu)
    return {"adjs_u_from_i": adjs_u_from_i, "adjs_i_from_u": adjs_i_from_u}


class CometLogger:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.enabled = bool(cfg.comet.enabled and COMET_AVAILABLE and os.getenv("COMET_API_KEY"))
        self.exp = None
        if self.enabled:
            self.exp = Experiment(
                api_key=os.getenv("COMET_API_KEY"),
                project_name=cfg.comet.project_name,
                workspace=cfg.comet.workspace,
                log_code=cfg.comet.log_code,
            )
            if cfg.comet.experiment_name:
                self.exp.set_name(cfg.comet.experiment_name)
            if cfg.comet.tags:
                self.exp.add_tags(cfg.comet.tags)
            self.exp.log_parameters(dc_to_dict(cfg))
        else:
            reason = []
            if not cfg.comet.enabled:
                reason.append("disabled in config")
            if not COMET_AVAILABLE:
                reason.append("comet_ml import failed")
            if not os.getenv("COMET_API_KEY"):
                reason.append("COMET_API_KEY is not set")
            print(f"[comet] offline mode: {', '.join(reason)}")

    def log_metrics(self, metrics: dict, step: Optional[int] = None, epoch: Optional[int] = None, prefix: str = "") -> None:
        if prefix:
            metrics = {f"{prefix}{k}": v for k, v in metrics.items()}
        if self.exp is not None:
            self.exp.log_metrics(metrics, step=step, epoch=epoch)

    def log_text(self, name: str, text: str) -> None:
        if self.exp is not None:
            self.exp.log_text(text, metadata={"name": name})

    def end(self):
        if self.exp is not None:
            self.exp.end()


In [ ]:
def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor, sample_weights: Optional[torch.Tensor] = None) -> torch.Tensor:
    loss = -F.logsigmoid(pos_scores - neg_scores)
    if sample_weights is not None:
        loss = loss * sample_weights
    return loss.mean()


def to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(device, non_blocking=(device.type == "cuda"))
        else:
            out[k] = v
    return out


def compute_support_alignment_loss(
    model: SupportAwareGCMCModel,
    data: PreparedData,
    cfg: ExperimentConfig,
    device: torch.device,
    user_z: torch.Tensor,
    item_z: torch.Tensor,
) -> torch.Tensor:
    if cfg.model.graph.support_alignment_weight <= 0:
        return user_z.new_tensor(0.0)
    train_users = sorted(data.user_histories.keys())
    if not train_users:
        return user_z.new_tensor(0.0)
    sample_size = min(len(train_users), int(cfg.model.graph.support_user_batch_size))
    if sample_size <= 0:
        return user_z.new_tensor(0.0)
    if sample_size < len(train_users):
        sampled_users = random.sample(train_users, sample_size)
    else:
        sampled_users = train_users
    hist_batch = build_eval_history_tensors(sampled_users, data.user_histories, cfg.train.max_history, device)
    support_user_z = model.encode_support_users(**hist_batch, item_vectors=item_z)
    target_user_z = user_z[torch.tensor(sampled_users, dtype=torch.long, device=device)]
    loss_kind = cfg.model.graph.support_alignment_loss
    if loss_kind == "cosine":
        return (1.0 - F.cosine_similarity(support_user_z, target_user_z, dim=-1)).mean()
    return F.mse_loss(support_user_z, target_user_z)


def train_one_epoch_pairwise(model, loader, optimizer, cfg: ExperimentConfig, logger: CometLogger, epoch: int) -> Dict[str, float]:
    device = torch.device(cfg.train.device)
    model.train()
    scaler = make_grad_scaler(cfg, device)
    total_loss = 0.0
    total_steps = 0
    total_examples = 0
    step_times = []
    grad_norms = []
    pbar = tqdm(loader, desc=f"Train epoch {epoch}", disable=not cfg.train.verbose)
    for step, batch in enumerate(pbar, start=1):
        step_start = time.perf_counter()
        batch = to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        with get_autocast_context(cfg, device):
            if cfg.model.model_type == "mf":
                pos_scores, neg_scores = model(batch["user_idx"], batch["pos_item_idx"], batch["neg_item_idx"])
            elif cfg.model.model_type == "two_tower":
                pos_scores, neg_scores = model(
                    batch["user_idx"],
                    batch["pos_item_idx"],
                    batch["neg_item_idx"],
                    batch["hist_items"],
                    batch["hist_ratings"],
                    batch["hist_timestamps"],
                    batch["hist_mask"],
                    batch["target_timestamp"],
                )
            else:
                raise ValueError(f"train_one_epoch_pairwise is not for model_type={cfg.model.model_type}")

            sample_w = rating_weight(batch["rating"], cfg.train.train_rating_weighting)
            loss = bpr_loss(pos_scores, neg_scores, sample_w)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = compute_grad_norm(model.parameters())
        if cfg.train.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item())
        total_steps += 1
        total_examples += int(batch["user_idx"].shape[0])
        step_time = time.perf_counter() - step_start
        step_times.append(step_time)
        grad_norms.append(grad_norm)
        pbar.set_postfix(loss=f"{loss.item():.4f}", grad=f"{grad_norm:.3f}")

        if step % cfg.train.log_every_n_steps == 0:
            logger.log_metrics(
                {
                    "train/loss_step": float(loss.item()),
                    "train/step_time_sec": float(step_time),
                    "train/grad_norm_step": float(grad_norm),
                    "train/examples_per_sec": float(batch["user_idx"].shape[0] / max(step_time, 1e-8)),
                },
                step=(epoch - 1) * len(loader) + step,
                epoch=epoch,
            )

    return {
        "train_loss": float(total_loss / max(total_steps, 1)),
        "train_step_time_mean_sec": float(np.mean(step_times)) if step_times else 0.0,
        "train_grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else 0.0,
        "train_grad_norm_max": float(np.max(grad_norms)) if grad_norms else 0.0,
        "train_examples_per_sec": float(total_examples / max(np.sum(step_times), 1e-8)) if step_times else 0.0,
        "train_steps": int(total_steps),
    }


def train_one_epoch_relation_gcmc(
    model: RelationGCMCModel,
    edge_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    cfg: ExperimentConfig,
    data: PreparedData,
    logger: CometLogger,
    epoch: int,
) -> Dict[str, float]:
    device = torch.device(cfg.train.device)
    model.train()
    scaler = make_grad_scaler(cfg, device)
    optimizer.zero_grad(set_to_none=True)
    forward_start = time.perf_counter()
    with get_autocast_context(cfg, device):
        user_z, item_z = model.encode_full_graph()
    forward_time = time.perf_counter() - forward_start

    total_loss = 0.0
    total_edges = 0
    num_batches = 0
    loss_accum = None
    align_loss_value = 0.0

    pbar = tqdm(edge_loader, desc=f"Graph loss epoch {epoch}", disable=not cfg.train.verbose)
    for step, batch in enumerate(pbar, start=1):
        batch = to_device(batch, device)
        with get_autocast_context(cfg, device):
            logits = model.decode(user_z[batch["user_idx"]], item_z[batch["item_idx"]])
            ce = F.cross_entropy(logits, batch["rating_class"], reduction="mean")
            if cfg.model.graph.aux_bpr_weight > 0:
                raise NotImplementedError("Auxiliary BPR for relation_gcmc is left for a future extension.")
            loss = ce
        batch_weight = batch["user_idx"].shape[0] / len(edge_loader.dataset)
        loss_accum = loss * batch_weight if loss_accum is None else loss_accum + loss * batch_weight
        total_loss += float(loss.item()) * batch["user_idx"].shape[0]
        total_edges += int(batch["user_idx"].shape[0])
        num_batches += 1

        if step % max(1, cfg.train.graph_loss_batches_per_epoch_log) == 0:
            logger.log_metrics(
                {
                    "train/relation_batch_ce": float(loss.item()),
                },
                step=(epoch - 1) * len(edge_loader) + step,
                epoch=epoch,
            )
        pbar.set_postfix(ce=f"{loss.item():.4f}")

    base_graph_model = unwrap_model(model)
    if hasattr(base_graph_model, "encode_support_users") and cfg.model.graph.support_alignment_weight > 0:
        with get_autocast_context(cfg, device):
            align_loss = compute_support_alignment_loss(base_graph_model, data, cfg, device, user_z, item_z)
        align_loss_value = float(align_loss.item())
        loss_accum = loss_accum + cfg.model.graph.support_alignment_weight * align_loss

    backward_start = time.perf_counter()
    scaler.scale(loss_accum).backward()
    scaler.unscale_(optimizer)
    grad_norm = compute_grad_norm(model.parameters())
    if cfg.train.grad_clip_norm is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip_norm)
    scaler.step(optimizer)
    scaler.update()
    backward_time = time.perf_counter() - backward_start

    return {
        "train_loss": float(total_loss / max(total_edges, 1)),
        "graph_forward_time_sec": float(forward_time),
        "graph_backward_time_sec": float(backward_time),
        "train_grad_norm_mean": float(grad_norm),
        "train_grad_norm_max": float(grad_norm),
        "train_edges_per_sec": float(total_edges / max(forward_time + backward_time, 1e-8)),
        "support_alignment_loss": float(align_loss_value),
        "train_batches": int(num_batches),
    }


def build_eval_history_tensors(
    user_indices: Sequence[int],
    user_histories: Dict[int, List[Tuple[int, float, int]]],
    max_history: int,
    device: torch.device,
) -> dict:
    batch_hist = [user_histories.get(int(u), [])[-max_history:] for u in user_indices]
    max_len = max((len(h) for h in batch_hist), default=1)
    hist_items = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_ratings = torch.zeros((len(user_indices), max_len), dtype=torch.float32, device=device)
    hist_timestamps = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_mask = torch.zeros((len(user_indices), max_len), dtype=torch.bool, device=device)
    target_timestamp = torch.zeros((len(user_indices),), dtype=torch.long, device=device)

    for row, hist in enumerate(batch_hist):
        if not hist:
            continue
        items, ratings, timestamps = zip(*hist)
        n = len(items)
        hist_items[row, :n] = torch.tensor(items, dtype=torch.long, device=device)
        hist_ratings[row, :n] = torch.tensor(ratings, dtype=torch.float32, device=device)
        hist_timestamps[row, :n] = torch.tensor(timestamps, dtype=torch.long, device=device)
        hist_mask[row, :n] = True
        target_timestamp[row] = int(max(timestamps))
    return {
        "hist_items": hist_items,
        "hist_ratings": hist_ratings,
        "hist_timestamps": hist_timestamps,
        "hist_mask": hist_mask,
        "target_timestamp": target_timestamp,
    }


def build_eval_history_tensors_for_rows(
    user_indices: Sequence[int],
    target_timestamps: Sequence[int],
    user_histories: Dict[int, List[Tuple[int, float, int]]],
    max_history: int,
    device: torch.device,
) -> dict:
    batch_hist = [user_histories.get(int(u), [])[-max_history:] for u in user_indices]
    max_len = max((len(h) for h in batch_hist), default=1)
    hist_items = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_ratings = torch.zeros((len(user_indices), max_len), dtype=torch.float32, device=device)
    hist_timestamps = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_mask = torch.zeros((len(user_indices), max_len), dtype=torch.bool, device=device)
    target_timestamp = torch.tensor(list(target_timestamps), dtype=torch.long, device=device)

    for row, hist in enumerate(batch_hist):
        if not hist:
            continue
        items, ratings, timestamps = zip(*hist)
        n = len(items)
        hist_items[row, :n] = torch.tensor(items, dtype=torch.long, device=device)
        hist_ratings[row, :n] = torch.tensor(ratings, dtype=torch.float32, device=device)
        hist_timestamps[row, :n] = torch.tensor(timestamps, dtype=torch.long, device=device)
        hist_mask[row, :n] = True
    return {
        "hist_items": hist_items,
        "hist_ratings": hist_ratings,
        "hist_timestamps": hist_timestamps,
        "hist_mask": hist_mask,
        "target_timestamp": target_timestamp,
    }


def _stable_seed_from_text(text: str, base_seed: int = 42) -> int:
    seed = int(base_seed) & 0xFFFFFFFF
    for ch in text:
        seed = (seed * 131 + ord(ch)) & 0xFFFFFFFF
    return seed


def _sample_negative_item(user_seen: set, num_items: int, rng: np.random.Generator) -> int:
    if len(user_seen) >= num_items:
        return int(rng.integers(0, num_items))
    neg = int(rng.integers(0, num_items))
    max_tries = 64
    tries = 0
    while neg in user_seen and tries < max_tries:
        neg = int(rng.integers(0, num_items))
        tries += 1
    if neg in user_seen:
        candidates = np.setdiff1d(np.arange(num_items, dtype=np.int64), np.fromiter(user_seen, dtype=np.int64), assume_unique=False)
        if len(candidates) == 0:
            return int(rng.integers(0, num_items))
        neg = int(candidates[int(rng.integers(0, len(candidates)))])
    return neg


def compute_eval_loss(model, data: PreparedData, cfg: ExperimentConfig, eval_key: str) -> Dict[str, float]:
    if not cfg.eval.compute_loss:
        return {}
    if eval_key not in data.eval_frames or not model_supports_eval_key(cfg.model.model_type, eval_key):
        return {}

    eval_df = data.eval_frames[eval_key]
    if eval_df.empty:
        return {}

    if cfg.eval.loss_max_rows is not None and len(eval_df) > cfg.eval.loss_max_rows:
        eval_df = eval_df.sample(cfg.eval.loss_max_rows, random_state=cfg.train.seed).reset_index(drop=True)

    device = torch.device(cfg.train.device)
    base_model = unwrap_model(model)
    base_model.eval()
    user_histories = data.eval_histories.get(eval_key, data.user_histories)
    base_seen_items = data.eval_seen_items.get(eval_key, data.seen_items)
    observed_eval_items = defaultdict(set)
    for row in eval_df[["user_idx", "item_idx"]].itertuples(index=False):
        observed_eval_items[int(row.user_idx)].add(int(row.item_idx))
    exclusion_by_user = {
        int(u): set(base_seen_items.get(int(u), set())) | observed_eval_items.get(int(u), set())
        for u in set(base_seen_items.keys()) | set(observed_eval_items.keys())
    }

    if cfg.model.model_type in {"mf", "two_tower"}:
        pos_df = eval_df.loc[eval_df["rating"] >= cfg.train.positive_threshold, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if pos_df.empty:
            return {"loss": 0.0, "loss_rows": 0}
        rng = np.random.default_rng(_stable_seed_from_text(eval_key, cfg.train.seed))
        total_loss = 0.0
        total_rows = 0
        batch_size = max(1, cfg.eval.loss_batch_size)
        with torch.no_grad():
            for start in range(0, len(pos_df), batch_size):
                batch_df = pos_df.iloc[start: start + batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                pos_item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                ratings = batch_df["rating"].to_numpy(dtype=np.float32)
                timestamps = batch_df["timestamp"].to_numpy(dtype=np.int64)
                neg_item_idx = np.array([
                    _sample_negative_item(exclusion_by_user.get(int(u), set()), data.num_items, rng)
                    for u in user_idx
                ], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                pos_t = torch.tensor(pos_item_idx, dtype=torch.long, device=device)
                neg_t = torch.tensor(neg_item_idx, dtype=torch.long, device=device)
                rating_t = torch.tensor(ratings, dtype=torch.float32, device=device)

                with get_autocast_context(cfg, device):
                    if cfg.model.model_type == "mf":
                        pos_scores = base_model.score(user_t, pos_t)
                        neg_scores = base_model.score(user_t, neg_t)
                    else:
                        hist_batch = build_eval_history_tensors_for_rows(
                            user_indices=user_idx.tolist(),
                            target_timestamps=timestamps.tolist(),
                            user_histories=user_histories,
                            max_history=cfg.train.max_history,
                            device=device,
                        )
                        user_vec = base_model.encode_user(user_t, **hist_batch)
                        pos_scores = (user_vec * base_model.encode_item(pos_t)).sum(dim=-1)
                        neg_scores = (user_vec * base_model.encode_item(neg_t)).sum(dim=-1)
                    sample_w = rating_weight(rating_t, cfg.train.train_rating_weighting)
                    loss = bpr_loss(pos_scores, neg_scores, sample_w)
                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows
        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}

    if cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
        valid_mask = eval_df["rating"].isin(data.rating2class.keys())
        eval_obs = eval_df.loc[valid_mask, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if eval_obs.empty:
            return {"loss": 0.0, "loss_rows": 0}
        graph_batch_size = max(1, cfg.eval.loss_batch_size)
        total_loss = 0.0
        total_rows = 0
        with torch.no_grad():
            with get_autocast_context(cfg, device):
                graph_user_z, graph_item_z = base_model.encode_full_graph()
            for start in range(0, len(eval_obs), graph_batch_size):
                batch_df = eval_obs.iloc[start: start + graph_batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                timestamps = batch_df["timestamp"].to_numpy(dtype=np.int64)
                rating_class = np.array([data.rating2class[float(r)] for r in batch_df["rating"].tolist()], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                item_t = torch.tensor(item_idx, dtype=torch.long, device=device)
                class_t = torch.tensor(rating_class, dtype=torch.long, device=device)

                with get_autocast_context(cfg, device):
                    if cfg.model.model_type == "support_gcmc" and (eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_")):
                        hist_batch = build_eval_history_tensors_for_rows(
                            user_indices=user_idx.tolist(),
                            target_timestamps=timestamps.tolist(),
                            user_histories=user_histories,
                            max_history=cfg.train.max_history,
                            device=device,
                        )
                        user_vec = base_model.encode_support_users(**hist_batch, item_vectors=graph_item_z)
                    else:
                        user_vec = graph_user_z[user_t]
                    logits = base_model.decode(user_vec, graph_item_z[item_t])
                    loss = F.cross_entropy(logits, class_t, reduction="mean")
                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows
        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}

    return {}


def get_eval_metric_names(cfg_eval: EvalConfig, gain_type: str) -> Tuple[str, ...]:
    family = gain_type_metric_family(gain_type)
    if family == "binary":
        return tuple(cfg_eval.binary_metric_names)
    return tuple(cfg_eval.graded_metric_names)


def get_eval_keys(cfg: ExperimentConfig, split_name: str) -> List[str]:
    keys = [f"warm_{split_name}"]
    include_cold = cfg.eval.evaluate_cold_on_val if split_name == "val" else cfg.eval.evaluate_cold_on_test
    if not include_cold:
        return keys
    for scenario in cfg.eval.eval_scenarios:
        if scenario == "warm":
            continue
        keys.append(f"{scenario}_{split_name}")
    return keys


def model_supports_eval_key(model_type: str, eval_key: str) -> bool:
    if eval_key.startswith("warm_"):
        return True
    if eval_key.startswith("cold_item_"):
        return model_type in {"mf", "two_tower", "relation_gcmc", "support_gcmc"}
    if eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_"):
        return model_type in {"two_tower", "support_gcmc"}
    return False


import numpy as np
from typing import Dict, Sequence


def dcg_at_k(gains: Sequence[float], k: int) -> float:
    gains = np.asarray(gains[:k], dtype=np.float32)
    if gains.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))


def compute_ranking_metrics(
    ranked_items: np.ndarray,
    target_gain_dict: Dict[int, float],
    ks: Sequence[int],
    metric_names: Sequence[str],
) -> Dict[str, float]:
    metrics: Dict[str, float] = {}

    gains_ranked = np.asarray(
        [target_gain_dict.get(int(i), 0.0) for i in ranked_items],
        dtype=np.float32,
    )
    binary_ranked = (gains_ranked > 0).astype(np.float32)
    relevant_total = int(
        np.sum(np.asarray(list(target_gain_dict.values()), dtype=np.float32) > 0)
    )
    ideal_gains = sorted(target_gain_dict.values(), reverse=True)

    for k in ks:
        topk_gains = gains_ranked[:k]
        topk_binary = binary_ranked[:k]
        hits = int(np.sum(topk_binary))

        if "hitrate" in metric_names:
            metrics[f"hitrate@{k}"] = float(hits > 0)

        if "precision" in metric_names:
            metrics[f"precision@{k}"] = float(hits / max(k, 1))

        if "recall" in metric_names:
            metrics[f"recall@{k}"] = float(hits / relevant_total) if relevant_total > 0 else 0.0

        if "ndcg" in metric_names:
            dcg = dcg_at_k(topk_gains, k)
            idcg = dcg_at_k(ideal_gains, min(k, len(ideal_gains)))
            metrics[f"ndcg@{k}"] = float(dcg / idcg) if idcg > 0 else 0.0

        if "mrr" in metric_names or "map" in metric_names:
            rel_positions = np.flatnonzero(topk_binary > 0)

            if "mrr" in metric_names:
                metrics[f"mrr@{k}"] = float(1.0 / (rel_positions[0] + 1)) if rel_positions.size > 0 else 0.0

            if "map" in metric_names:
                if rel_positions.size > 0:
                    prefix_hits = np.cumsum(topk_binary)
                    precisions = prefix_hits[rel_positions] / (rel_positions + 1)
                    ap_denom = min(relevant_total, k)
                    metrics[f"map@{k}"] = float(np.sum(precisions) / ap_denom) if ap_denom > 0 else 0.0
                else:
                    metrics[f"map@{k}"] = 0.0

    return metrics


def evaluate_ranked_items(
    ranked_items: np.ndarray,
    target_gain_by_item: Dict[int, float],
    ks: Sequence[int],
    metric_names: Sequence[str],
) -> Dict[str, float]:
    return compute_ranking_metrics(
        ranked_items=ranked_items,
        target_gain_dict=target_gain_by_item,
        ks=ks,
        metric_names=metric_names,
    )





@torch.inference_mode()
def evaluate_model(model, data: PreparedData, cfg: ExperimentConfig, eval_key: str = "warm_val") -> Dict[str, float]:
    if eval_key not in data.eval_frames:
        return {"skipped": 1.0}
    if not model_supports_eval_key(cfg.model.model_type, eval_key):
        return {"skipped": 1.0}

    base_model = unwrap_model(model)
    device = torch.device(cfg.train.device)
    base_model.eval()
    eval_df = data.eval_frames[eval_key]
    if eval_df.empty:
        return {"users_scored": 0, "time_sec": 0.0, "empty": 1.0}

    gain_types = get_metric_gain_types(cfg.eval)
    targets_by_gain = {gain_type: build_eval_targets(eval_df, cfg.eval, gain_type) for gain_type in gain_types}
    users = sorted(set().union(*[set(v.keys()) for v in targets_by_gain.values()]))
    if not users:
        return {}

    metric_rows_by_gain: Dict[str, List[Dict[str, float]]] = {gain_type: [] for gain_type in gain_types}
    user_histories = data.eval_histories.get(eval_key, data.user_histories)
    seen_items_by_user = data.eval_seen_items.get(eval_key, data.seen_items)
    t0 = time.perf_counter()

    graph_user_z = None
    graph_item_z = None
    if cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
        with get_autocast_context(cfg, device):
            graph_user_z, graph_item_z = base_model.encode_full_graph()

    for start_idx in tqdm(range(0, len(users), cfg.eval.user_batch_size), desc=f"Eval {eval_key}"):
        batch_users = users[start_idx: start_idx + cfg.eval.user_batch_size]
        user_tensor = torch.tensor(batch_users, dtype=torch.long, device=device)

        with get_autocast_context(cfg, device):
            if cfg.model.model_type == "mf":
                score_matrix = base_model.score_all_items(user_tensor)
            elif cfg.model.model_type == "two_tower":
                hist_batch = build_eval_history_tensors(batch_users, user_histories, cfg.train.max_history, device)
                user_vecs = base_model.encode_user(user_tensor, **hist_batch)
                score_matrix = base_model.score_all_items_for_histories(user_vecs)
            elif cfg.model.model_type == "relation_gcmc":
                score_matrix = base_model.score_user_vectors_all_items(graph_user_z[user_tensor], graph_item_z, positive_threshold=cfg.eval.positive_threshold)
            elif cfg.model.model_type == "support_gcmc":
                if eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_"):
                    hist_batch = build_eval_history_tensors(batch_users, user_histories, cfg.train.max_history, device)
                    user_vecs = base_model.encode_support_users(**hist_batch, item_vectors=graph_item_z)
                else:
                    user_vecs = graph_user_z[user_tensor]
                score_matrix = base_model.score_user_vectors_all_items(user_vecs, graph_item_z, positive_threshold=cfg.eval.positive_threshold)
            else:
                raise ValueError(f"Unsupported model_type={cfg.model.model_type}")

        score_matrix = score_matrix.detach().float().cpu().numpy().astype(np.float32)
        max_k = max(cfg.eval.ks)

        for row_idx, user_idx in enumerate(batch_users):
            scores = score_matrix[row_idx]
            if cfg.eval.exclude_seen_items:
                seen = seen_items_by_user.get(int(user_idx), set())
                if seen:
                    seen_idx = np.fromiter(seen, dtype=np.int64)
                    valid_seen = seen_idx[(seen_idx >= 0) & (seen_idx < len(scores))]
                    if valid_seen.size > 0:
                        scores[valid_seen] = -1e9
            kth = min(max_k, len(scores) - 1)
            topk_idx = np.argpartition(-scores, kth=kth)[:max_k]
            ranked_items = topk_idx[np.argsort(-scores[topk_idx])]

            for gain_type in gain_types:
                target_gain = targets_by_gain[gain_type].get(int(user_idx), {})
                if not target_gain and cfg.eval.skip_users_with_no_relevant_targets:
                    continue
                metric_names = get_eval_metric_names(cfg.eval, gain_type)
                row_metrics = evaluate_ranked_items(
                    ranked_items=ranked_items,
                    target_gain_by_item=target_gain,
                    ks=cfg.eval.ks,
                    metric_names=metric_names,
                )
                metric_rows_by_gain[gain_type].append(row_metrics)

    elapsed = time.perf_counter() - t0
    out = {"users_scored": len(users), "time_sec": float(elapsed)}
    loss_metrics = compute_eval_loss(base_model, data, cfg, eval_key)
    out.update(loss_metrics)
    for gain_type, rows in metric_rows_by_gain.items():
        if not rows:
            continue
        df = pd.DataFrame(rows)
        for col in df.columns:
            out[f"{gain_type}/{col}"] = float(df[col].mean())
    return out


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


def unwrap_model(model: torch.nn.Module) -> torch.nn.Module:
    return model.module if hasattr(model, "module") else model


def maybe_wrap_for_multi_gpu(model: torch.nn.Module, cfg) -> torch.nn.Module:
    use_multi_gpu = bool(getattr(cfg.train, "use_multi_gpu", False))
    if not use_multi_gpu:
        return model
    if not torch.cuda.is_available():
        return model
    if torch.cuda.device_count() < 2:
        return model

    # Реально оборачиваем только те модели, для которых в ноутбуке
    # предполагался multi-GPU режим.
    if getattr(cfg.model, "model_type", None) in {"mf", "two_tower"}:
        return torch.nn.DataParallel(model)

    return model


def maybe_compile_model(model: torch.nn.Module, cfg) -> torch.nn.Module:
    use_compile = bool(
        getattr(cfg.train, "use_torch_compile", False)
        or getattr(cfg.train, "compile_model", False)
    )
    if not use_compile:
        return model

    if not hasattr(torch, "compile"):
        return model

    try:
        return torch.compile(model)
    except Exception as e:
        print(f"[warn] torch.compile disabled: {e}")
        return model


def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    cfg,
    collate_fn=None,
) -> DataLoader:
    num_workers = max(int(getattr(cfg.train, "num_workers", 0)), 0)

    loader_kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=bool(getattr(cfg.train, "pin_memory", torch.cuda.is_available())),
        collate_fn=collate_fn,
    )

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = bool(
            getattr(cfg.train, "persistent_workers", False)
        )
        prefetch_factor = getattr(cfg.train, "prefetch_factor", None)
        if prefetch_factor is not None:
            loader_kwargs["prefetch_factor"] = int(prefetch_factor)

    return DataLoader(**loader_kwargs)

In [ ]:
def build_model_and_loaders(cfg: ExperimentConfig, data: PreparedData):
    device = torch.device(cfg.train.device)
    warm_candidate_items = np.sort(data.train_df["item_idx"].astype(np.int64).unique())
    pairwise_ds = PairwiseTrainDataset(data.train_df, data.seen_items, warm_candidate_items, cfg.train.positive_threshold)
    pairwise_loader = make_loader(pairwise_ds, cfg.train.batch_size, True, cfg)

    if cfg.model.model_type == "mf":
        model = MFModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.mf,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        model = maybe_wrap_for_multi_gpu(model, cfg)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, pairwise_loader

    if cfg.model.model_type == "two_tower":
        seq_ds = SequencePairwiseDataset(
            data.train_df,
            warm_candidate_items,
            cfg.train.positive_threshold,
            cfg.train.max_history,
            cfg.train.min_history_for_sequence,
        )
        seq_loader = make_loader(seq_ds, cfg.train.batch_size, True, cfg, collate_fn=sequence_collate_fn)
        model = TwoTowerModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.two_tower,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        model = maybe_wrap_for_multi_gpu(model, cfg)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, seq_loader

    if cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
        edge_ds = ObservedEdgeDataset(data.train_df, data.rating2class)
        edge_loader = make_loader(edge_ds, cfg.train.relation_edge_batch_size, True, cfg)
        model_cls = SupportAwareGCMCModel if cfg.model.model_type == "support_gcmc" else RelationGCMCModel
        model = model_cls(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.graph,
            data.unique_ratings,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        graph_dict = build_relation_graph(
            data.train_df,
            data.num_users,
            data.num_items,
            data.unique_ratings,
            device,
            apply_time_decay=cfg.model.graph.relation_apply_time_decay,
            time_weight_fn_name=cfg.model.graph.time_weight_fn,
            gamma=cfg.model.graph.time_weight_gamma,
        )
        model.set_graph(graph_dict)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, edge_loader

    raise ValueError(f"Unknown model_type={cfg.model.model_type}")


In [ ]:
def select_main_metric(metrics: Dict[str, float], cfg_eval: EvalConfig) -> float:
    metric_k = cfg_eval.main_metric_k if cfg_eval.main_metric_k in cfg_eval.ks else cfg_eval.ks[0]
    key = f"{cfg_eval.main_metric_gain_type}/{cfg_eval.main_metric_name}@{metric_k}"
    return float(metrics.get(key, -1e9))


def save_checkpoint(model: nn.Module, output_dir: Path, name: str) -> None:
    path = output_dir / name
    torch.save(unwrap_model(model).state_dict(), path)
    print(f"[checkpoint] saved to {path}")


def run_training(cfg: ExperimentConfig):
    set_seed(cfg.train.seed)
    configure_runtime(cfg)
    data = load_prepared_data(cfg)
    model, optimizer, train_loader = build_model_and_loaders(cfg, data)
    logger = CometLogger(cfg)
    runtime_info = get_gpu_info()
    logger.log_metrics({
        "runtime/num_gpus": runtime_info["num_gpus"],
        "runtime/use_amp": float(cfg.train.use_amp),
        "runtime/use_multi_gpu": float(cfg.train.use_multi_gpu),
    })
    if runtime_info["gpu_names"]:
        logger.log_text("gpu_names", ", ".join(runtime_info["gpu_names"]))
    save_json(dc_to_dict(cfg), data.output_dir / f"config_{cfg.model.model_type}.json")

    best_metric = -1e18
    best_epoch = -1
    best_metrics = None
    patience_counter = 0
    history = []

    val_eval_keys = get_eval_keys(cfg, "val")
    test_eval_keys = get_eval_keys(cfg, "test")

    for epoch in range(1, cfg.train.epochs + 1):
        print("=" * 80)
        print(f"Epoch {epoch}/{cfg.train.epochs} | model_type={cfg.model.model_type}")
        epoch_start = time.perf_counter()

        if cfg.model.model_type in {"mf", "two_tower"}:
            train_metrics = train_one_epoch_pairwise(model, train_loader, optimizer, cfg, logger, epoch)
        elif cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
            train_metrics = train_one_epoch_relation_gcmc(model, train_loader, optimizer, cfg, data, logger, epoch)
        else:
            raise ValueError(cfg.model.model_type)

        val_by_scenario: Dict[str, Dict[str, float]] = {}
        flat_val_metrics: Dict[str, float] = {}
        for eval_key in val_eval_keys:
            scenario_metrics = evaluate_model(model, data, cfg, eval_key=eval_key)
            val_by_scenario[eval_key] = scenario_metrics
            flat_val_metrics.update({f"val/{eval_key}/{k}": v for k, v in scenario_metrics.items()})

        epoch_time = time.perf_counter() - epoch_start
        epoch_metrics = {**train_metrics, **flat_val_metrics, "epoch_time_sec": float(epoch_time)}
        logger.log_metrics(epoch_metrics, epoch=epoch)
        history.append({"epoch": epoch, **epoch_metrics})

        print("TRAIN | " + " | ".join(f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}" for k, v in train_metrics.items()))
        for eval_key, scenario_metrics in val_by_scenario.items():
            printable = " | ".join(f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}" for k, v in scenario_metrics.items())
            print(f"VAL[{eval_key}] | {printable}")
        print(f"TIME  | epoch_time_sec={epoch_time:.3f}")

        current_metric = select_main_metric(val_by_scenario.get("warm_val", {}), cfg.eval)
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_metrics = val_by_scenario.get("warm_val", {})
            patience_counter = 0
            save_checkpoint(model, data.output_dir, f"best_{cfg.model.model_type}.pt")
        else:
            patience_counter += 1
            print(f"[early-stop] no improvement for {patience_counter} epoch(s)")
            if patience_counter >= cfg.train.patience:
                print("[early-stop] stopping training")
                break

    print("=" * 80)
    print(f"Best epoch={best_epoch} | main_metric={best_metric:.6f}")
    hist_df = pd.DataFrame(history)
    hist_path = data.output_dir / f"history_{cfg.model.model_type}.csv"
    hist_df.to_csv(hist_path, index=False)
    print(f"Saved training history to: {hist_path}")

    best_ckpt = data.output_dir / f"best_{cfg.model.model_type}.pt"
    if best_ckpt.exists():
        unwrap_model(model).load_state_dict(torch.load(best_ckpt, map_location=cfg.train.device))

    test_by_scenario: Dict[str, Dict[str, float]] = {}
    flat_test_metrics: Dict[str, float] = {}
    for eval_key in test_eval_keys:
        scenario_metrics = evaluate_model(model, data, cfg, eval_key=eval_key)
        test_by_scenario[eval_key] = scenario_metrics
        flat_test_metrics.update({f"test/{eval_key}/{k}": v for k, v in scenario_metrics.items()})
    logger.log_metrics(flat_test_metrics)
    logger.end()

    return model, data, hist_df, best_metrics, test_by_scenario


## Hybrid implementation

The next cell adds the three composite models and overrides the training/evaluation dispatchers so the old base models still run unchanged.

In [ ]:

# =========================
# Hybrid extensions
# =========================
# This cell is intentionally self-contained: it keeps the original TwoTowerModel,
# SupportAwareGCMCModel, RelationGCMLayer and BilinearRatingDecoder unchanged,
# and builds wrappers around them.

@dataclass
class HybridConfig:
    # Paths are optional. If None or missing, the corresponding base model starts from random init.
    # For serious runs: first train/save best_two_tower.pt and best_relation_gcmc.pt or best_support_gcmc.pt.
    graph_checkpoint_path: Optional[str] = None
    two_tower_checkpoint_path: Optional[str] = None

    # Practical default for Kaggle: staged/frozen encoders.
    # End-to-end graph recomputation for every pairwise batch is expensive on MovieLens-20M.
    freeze_graph_encoder: bool = True
    freeze_two_tower_encoder: bool = True

    # Small trainable projections after frozen base embeddings.
    use_projection_adapters: bool = True

    # Weighted ensemble parameters.
    fusion_alpha_init: float = 0.50
    separate_user_item_alpha: bool = True

    # If True, missing checkpoint path raises an error instead of warning.
    require_pretrained: bool = False


@dataclass
class ModelConfig:
    # Supported:
    #   mf | two_tower | relation_gcmc | support_gcmc
    #   gcmc_to_two_tower | two_tower_to_gcmc | weighted_tt_gcmc_ensemble
    model_type: str = "mf"
    mf: MFConfig = field(default_factory=MFConfig)
    two_tower: TwoTowerConfig = field(default_factory=TwoTowerConfig)
    graph: GraphConfig = field(default_factory=GraphConfig)
    hybrid: HybridConfig = field(default_factory=HybridConfig)


@dataclass
class ExperimentConfig:
    paths: PathsConfig = field(default_factory=PathsConfig)
    features: FeatureConfig = field(default_factory=FeatureConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)
    comet: CometConfig = field(default_factory=CometConfig)
    model: ModelConfig = field(default_factory=ModelConfig)


PAIRWISE_MODEL_TYPES = {"mf", "two_tower", "gcmc_to_two_tower", "weighted_tt_gcmc_ensemble"}
GRAPH_MODEL_TYPES = {"relation_gcmc", "support_gcmc", "two_tower_to_gcmc"}
HYBRID_MODEL_TYPES = {"gcmc_to_two_tower", "two_tower_to_gcmc", "weighted_tt_gcmc_ensemble"}


def _strip_module_prefix(state_dict: dict) -> dict:
    if state_dict and all(str(k).startswith("module.") for k in state_dict.keys()):
        return {str(k)[7:]: v for k, v in state_dict.items()}
    return state_dict


def load_state_dict_if_available(
    model: nn.Module,
    checkpoint_path: Optional[str],
    *,
    strict: bool = False,
    require: bool = False,
    name: str = "model",
) -> None:
    if checkpoint_path is None or str(checkpoint_path).strip() == "":
        msg = f"[checkpoint] no checkpoint path provided for {name}; using current initialization"
        if require:
            raise FileNotFoundError(msg)
        print(msg)
        return

    ckpt_path = Path(checkpoint_path)
    if not ckpt_path.exists():
        msg = f"[checkpoint] checkpoint for {name} not found: {ckpt_path}"
        if require:
            raise FileNotFoundError(msg)
        print(msg + "; using current initialization")
        return

    state = torch.load(ckpt_path, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
        state = state["state_dict"]
    state = _strip_module_prefix(state)
    missing, unexpected = model.load_state_dict(state, strict=strict)
    print(
        f"[checkpoint] loaded {name} from {ckpt_path} | "
        f"missing={len(missing)} unexpected={len(unexpected)} strict={strict}"
    )
    if missing:
        print("  missing sample:", list(missing)[:8])
    if unexpected:
        print("  unexpected sample:", list(unexpected)[:8])


def freeze_module(module: nn.Module, freeze: bool = True) -> None:
    for p in module.parameters():
        p.requires_grad_(not freeze)


def make_projection_adapter(dim: int, dropout: float, use_projection: bool) -> nn.Module:
    if not use_projection:
        return nn.Identity()
    return nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, dim),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(dim, dim),
    )


def make_graph_model_for_hybrid(cfg: ExperimentConfig, data: PreparedData, device: torch.device) -> SupportAwareGCMCModel:
    graph_model = SupportAwareGCMCModel(
        data.num_users,
        data.num_items,
        data.item_features,
        cfg.features,
        cfg.model.graph,
        data.unique_ratings,
        trainable_item_mask=data.trainable_item_mask,
    ).to(device)
    graph_dict = build_relation_graph(
        data.train_df,
        data.num_users,
        data.num_items,
        data.unique_ratings,
        device,
        apply_time_decay=cfg.model.graph.relation_apply_time_decay,
        time_weight_fn_name=cfg.model.graph.time_weight_fn,
        gamma=cfg.model.graph.time_weight_gamma,
    )
    graph_model.set_graph(graph_dict)
    return graph_model


def make_two_tower_model_for_hybrid(cfg: ExperimentConfig, data: PreparedData, device: torch.device) -> TwoTowerModel:
    return TwoTowerModel(
        data.num_users,
        data.num_items,
        data.item_features,
        cfg.features,
        cfg.model.two_tower,
        trainable_item_mask=data.trainable_item_mask,
    ).to(device)


class GraphCacheMixin:
    def _init_graph_cache(self):
        self._graph_user_z = None
        self._graph_item_z = None

    def refresh_graph_cache(self) -> None:
        if not hasattr(self, "graph_model"):
            return
        if self.freeze_graph_encoder:
            was_training = self.graph_model.training
            self.graph_model.eval()
            with torch.no_grad():
                user_z, item_z = self.graph_model.encode_full_graph()
            if was_training:
                self.graph_model.train()
            user_z = user_z.detach()
            item_z = item_z.detach()
        else:
            # This is supported for evaluation/small experiments, but default training keeps graph frozen.
            user_z, item_z = self.graph_model.encode_full_graph()

        self._graph_user_z = user_z
        self._graph_item_z = item_z

    def graph_cache_ready(self) -> bool:
        return self._graph_user_z is not None and self._graph_item_z is not None

    def require_graph_cache(self) -> None:
        if not self.graph_cache_ready():
            self.refresh_graph_cache()


class GCMCToTwoTowerModel(nn.Module, GraphCacheMixin):
    """
    GCMC -> Two-Tower:
    - GCMC produces user/item node embeddings.
    - The Two-Tower user tower keeps the same SASRecSequenceEncoder, rating/time projections and user MLP.
    - History tokens and target item vectors come from graph item embeddings instead of FeatureAwareItemEncoder.
    """

    def __init__(
        self,
        graph_model: SupportAwareGCMCModel,
        two_tower_cfg: TwoTowerConfig,
        hybrid_cfg: HybridConfig,
    ):
        super().__init__()
        self.graph_model = graph_model
        self.model_cfg = two_tower_cfg
        self.hybrid_cfg = hybrid_cfg
        self.embedding_dim = two_tower_cfg.embedding_dim
        self.num_users = graph_model.num_users
        self.num_items = graph_model.item_encoder.num_items
        self.freeze_graph_encoder = bool(hybrid_cfg.freeze_graph_encoder)
        freeze_module(self.graph_model, self.freeze_graph_encoder)

        self.item_adapter = make_projection_adapter(
            self.embedding_dim,
            two_tower_cfg.dropout,
            hybrid_cfg.use_projection_adapters,
        )
        self.user_adapter = make_projection_adapter(
            self.embedding_dim,
            two_tower_cfg.dropout,
            hybrid_cfg.use_projection_adapters,
        )

        self.use_user_id = two_tower_cfg.use_user_id_embedding
        self.rating_proj = nn.Sequential(
            nn.Linear(1, two_tower_cfg.embedding_dim),
            nn.GELU(),
            nn.Linear(two_tower_cfg.embedding_dim, two_tower_cfg.embedding_dim),
        )
        self.time_proj = nn.Sequential(
            nn.Linear(1, two_tower_cfg.embedding_dim),
            nn.GELU(),
            nn.Linear(two_tower_cfg.embedding_dim, two_tower_cfg.embedding_dim),
        )
        self.sequence_encoder = SASRecSequenceEncoder(
            embedding_dim=two_tower_cfg.embedding_dim,
            max_seq_len=two_tower_cfg.sasrec_max_seq_len,
            num_layers=two_tower_cfg.sasrec_num_layers,
            num_heads=two_tower_cfg.sasrec_num_heads,
            ffn_dim=two_tower_cfg.sasrec_ffn_dim,
            dropout=two_tower_cfg.dropout,
            attention_dropout=two_tower_cfg.sasrec_attention_dropout,
            use_causal_mask=two_tower_cfg.sasrec_use_causal_mask,
        )
        self.user_mlp = nn.Sequential(
            nn.Linear(two_tower_cfg.embedding_dim * (2 if self.use_user_id else 1), two_tower_cfg.hidden_dim),
            nn.GELU(),
            nn.Dropout(two_tower_cfg.dropout),
            nn.Linear(two_tower_cfg.hidden_dim, two_tower_cfg.embedding_dim),
        )
        self.dropout = nn.Dropout(two_tower_cfg.dropout)
        self._init_graph_cache()

    def encode_item(self, item_idx: torch.Tensor) -> torch.Tensor:
        self.require_graph_cache()
        return self.dropout(self.item_adapter(self._graph_item_z[item_idx]))

    def build_history_tokens(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_vecs = self.encode_item(hist_items)

        if self.model_cfg.use_rating_in_history:
            seq_vecs = seq_vecs + self.rating_proj(hist_ratings.unsqueeze(-1))

        if self.model_cfg.use_time_in_history and target_timestamp is not None:
            delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0).float()
            time_input = torch.log1p(delta).unsqueeze(-1)
            seq_vecs = seq_vecs + self.time_proj(time_input)

        token_weights = hist_mask.float()
        if self.model_cfg.use_rating_in_history:
            token_weights = token_weights * rating_weight(hist_ratings, self.model_cfg.rating_weight_fn)
        if self.model_cfg.use_time_in_history and target_timestamp is not None:
            delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0).float()
            token_weights = token_weights * time_weight(
                delta,
                self.model_cfg.time_weight_fn,
                self.model_cfg.time_weight_gamma,
            )

        return self.dropout(seq_vecs * token_weights.unsqueeze(-1))

    def encode_history(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_vecs = self.build_history_tokens(
            hist_items=hist_items,
            hist_ratings=hist_ratings,
            hist_timestamps=hist_timestamps,
            hist_mask=hist_mask,
            target_timestamp=target_timestamp,
        )
        return self.sequence_encoder(seq_vecs, hist_mask)

    def encode_user(
        self,
        user_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        self.require_graph_cache()
        parts = [
            self.encode_history(
                hist_items=hist_items,
                hist_ratings=hist_ratings,
                hist_timestamps=hist_timestamps,
                hist_mask=hist_mask,
                target_timestamp=target_timestamp,
            )
        ]
        if self.use_user_id:
            valid_user = ((user_idx >= 0) & (user_idx < self.num_users)).float().unsqueeze(-1)
            safe_user_idx = torch.where(
                (user_idx >= 0) & (user_idx < self.num_users),
                user_idx,
                torch.zeros_like(user_idx),
            )
            parts.append(self.user_adapter(self._graph_user_z[safe_user_idx]) * valid_user)

        user_vec = torch.cat(parts, dim=-1)
        return self.user_mlp(self.dropout(user_vec))

    def score_batch(self, batch: dict) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.forward(
            batch["user_idx"],
            batch["pos_item_idx"],
            batch["neg_item_idx"],
            batch["hist_items"],
            batch["hist_ratings"],
            batch["hist_timestamps"],
            batch["hist_mask"],
            batch["target_timestamp"],
        )

    def forward(
        self,
        user_idx: torch.Tensor,
        pos_item_idx: torch.Tensor,
        neg_item_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_vec = self.encode_user(
            user_idx,
            hist_items,
            hist_ratings,
            hist_timestamps,
            hist_mask,
            target_timestamp,
        )
        pos_vec = self.encode_item(pos_item_idx)
        neg_vec = self.encode_item(neg_item_idx)
        return (user_vec * pos_vec).sum(dim=-1), (user_vec * neg_vec).sum(dim=-1)

    def score_all_items_for_histories(self, user_vecs: torch.Tensor) -> torch.Tensor:
        item_idx = torch.arange(self.num_items, device=user_vecs.device)
        item_vecs = self.encode_item(item_idx)
        return user_vecs @ item_vecs.T


class TwoTowerToGCMCModel(SupportAwareGCMCModel):
    """
    Two-Tower -> GCMC:
    - The two-tower item encoder and warm user-id embedding initialize GCMC node states.
    - RelationGCMLayer and BilinearRatingDecoder are the same GCMC-style graph modules as before.
    """

    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        graph_cfg: GraphConfig,
        rating_values: List[float],
        two_tower_model: TwoTowerModel,
        hybrid_cfg: HybridConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__(
            num_users=num_users,
            num_items=num_items,
            feature_tensor=feature_tensor,
            feature_cfg=feature_cfg,
            graph_cfg=graph_cfg,
            rating_values=rating_values,
            trainable_item_mask=trainable_item_mask,
        )
        self.two_tower_model = two_tower_model
        self.hybrid_cfg = hybrid_cfg
        self.freeze_two_tower_encoder = bool(hybrid_cfg.freeze_two_tower_encoder)
        freeze_module(self.two_tower_model, self.freeze_two_tower_encoder)
        self.user_init_adapter = make_projection_adapter(
            graph_cfg.embedding_dim,
            graph_cfg.dropout,
            hybrid_cfg.use_projection_adapters,
        )
        self.item_init_adapter = make_projection_adapter(
            graph_cfg.embedding_dim,
            graph_cfg.dropout,
            hybrid_cfg.use_projection_adapters,
        )

    def _two_tower_initial_states(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.freeze_two_tower_encoder:
            was_training = self.two_tower_model.training
            self.two_tower_model.eval()
            with torch.no_grad():
                if self.two_tower_model.user_id_emb is not None:
                    user_h = self.two_tower_model.user_id_emb.weight.detach()
                else:
                    user_h = self.user_emb.weight.detach()
                item_h = self.two_tower_model.item_encoder.all_item_vectors().detach()
            if was_training:
                self.two_tower_model.train()
        else:
            if self.two_tower_model.user_id_emb is not None:
                user_h = self.two_tower_model.user_id_emb.weight
            else:
                user_h = self.user_emb.weight
            item_h = self.two_tower_model.item_encoder.all_item_vectors()

        return self.user_init_adapter(user_h), self.item_init_adapter(item_h)

    def encode_full_graph(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if self._cached_graph is None:
            raise RuntimeError("Graph is not set. Call model.set_graph(graph_dict) first.")
        adjs_u_from_i = self._cached_graph["adjs_u_from_i"]
        adjs_i_from_u = self._cached_graph["adjs_i_from_u"]

        user_h, item_h = self._two_tower_initial_states()
        user_states = [user_h]
        item_states = [item_h]
        for layer in self.layers:
            user_h, item_h = layer(user_h, item_h, adjs_u_from_i, adjs_i_from_u)
            user_states.append(user_h)
            item_states.append(item_h)

        user_z = torch.stack(user_states, dim=0).mean(dim=0)
        item_z = torch.stack(item_states, dim=0).mean(dim=0)
        return user_z, item_z


class WeightedTwoTowerGCMCEnsemble(nn.Module, GraphCacheMixin):
    """
    Weighted ensemble:
        z_user = alpha_user * z_user_two_tower + (1 - alpha_user) * z_user_gcmc
        z_item = alpha_item * z_item_two_tower + (1 - alpha_item) * z_item_gcmc
    Alpha is learned during pairwise training.
    """

    def __init__(
        self,
        two_tower_model: TwoTowerModel,
        graph_model: SupportAwareGCMCModel,
        hybrid_cfg: HybridConfig,
    ):
        super().__init__()
        self.two_tower_model = two_tower_model
        self.graph_model = graph_model
        self.hybrid_cfg = hybrid_cfg
        self.embedding_dim = two_tower_model.embedding_dim
        self.num_users = two_tower_model.num_users
        self.num_items = two_tower_model.item_encoder.num_items

        self.freeze_two_tower_encoder = bool(hybrid_cfg.freeze_two_tower_encoder)
        self.freeze_graph_encoder = bool(hybrid_cfg.freeze_graph_encoder)
        freeze_module(self.two_tower_model, self.freeze_two_tower_encoder)
        freeze_module(self.graph_model, self.freeze_graph_encoder)

        init = float(np.clip(hybrid_cfg.fusion_alpha_init, 1e-4, 1.0 - 1e-4))
        logit = math.log(init / (1.0 - init))
        self.logit_alpha_user = nn.Parameter(torch.tensor(logit, dtype=torch.float32))
        if hybrid_cfg.separate_user_item_alpha:
            self.logit_alpha_item = nn.Parameter(torch.tensor(logit, dtype=torch.float32))
        else:
            self.logit_alpha_item = self.logit_alpha_user

        self.tt_user_adapter = make_projection_adapter(self.embedding_dim, two_tower_model.model_cfg.dropout, hybrid_cfg.use_projection_adapters)
        self.tt_item_adapter = make_projection_adapter(self.embedding_dim, two_tower_model.model_cfg.dropout, hybrid_cfg.use_projection_adapters)
        self.g_user_adapter = make_projection_adapter(self.embedding_dim, two_tower_model.model_cfg.dropout, hybrid_cfg.use_projection_adapters)
        self.g_item_adapter = make_projection_adapter(self.embedding_dim, two_tower_model.model_cfg.dropout, hybrid_cfg.use_projection_adapters)
        self._init_graph_cache()

    def alpha_user(self) -> torch.Tensor:
        return torch.sigmoid(self.logit_alpha_user)

    def alpha_item(self) -> torch.Tensor:
        return torch.sigmoid(self.logit_alpha_item)

    def _graph_user_vectors(
        self,
        user_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        self.require_graph_cache()
        out = torch.zeros((user_idx.shape[0], self.embedding_dim), dtype=self._graph_item_z.dtype, device=user_idx.device)
        valid = (user_idx >= 0) & (user_idx < self.num_users)
        if valid.any():
            out[valid] = self._graph_user_z[user_idx[valid]]
        if (~valid).any():
            support_z = self.graph_model.encode_support_users(
                hist_items=hist_items[~valid],
                hist_ratings=hist_ratings[~valid],
                hist_timestamps=hist_timestamps[~valid],
                hist_mask=hist_mask[~valid],
                target_timestamp=target_timestamp[~valid] if target_timestamp is not None else None,
                item_vectors=self._graph_item_z,
            )
            out[~valid] = support_z
        return out

    def encode_user(
        self,
        user_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if self.freeze_two_tower_encoder:
            was_training = self.two_tower_model.training
            self.two_tower_model.eval()
            with torch.no_grad():
                tt_user = self.two_tower_model.encode_user(
                    user_idx, hist_items, hist_ratings, hist_timestamps, hist_mask, target_timestamp
                ).detach()
            if was_training:
                self.two_tower_model.train()
        else:
            tt_user = self.two_tower_model.encode_user(
                user_idx, hist_items, hist_ratings, hist_timestamps, hist_mask, target_timestamp
            )

        g_user = self._graph_user_vectors(user_idx, hist_items, hist_ratings, hist_timestamps, hist_mask, target_timestamp)
        a = self.alpha_user().to(dtype=tt_user.dtype, device=tt_user.device)
        return a * self.tt_user_adapter(tt_user) + (1.0 - a) * self.g_user_adapter(g_user)

    def encode_item(self, item_idx: torch.Tensor) -> torch.Tensor:
        self.require_graph_cache()
        if self.freeze_two_tower_encoder:
            was_training = self.two_tower_model.training
            self.two_tower_model.eval()
            with torch.no_grad():
                tt_item = self.two_tower_model.encode_item(item_idx).detach()
            if was_training:
                self.two_tower_model.train()
        else:
            tt_item = self.two_tower_model.encode_item(item_idx)

        g_item = self._graph_item_z[item_idx]
        a = self.alpha_item().to(dtype=tt_item.dtype, device=tt_item.device)
        return a * self.tt_item_adapter(tt_item) + (1.0 - a) * self.g_item_adapter(g_item)

    def score_batch(self, batch: dict) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.forward(
            batch["user_idx"],
            batch["pos_item_idx"],
            batch["neg_item_idx"],
            batch["hist_items"],
            batch["hist_ratings"],
            batch["hist_timestamps"],
            batch["hist_mask"],
            batch["target_timestamp"],
        )

    def forward(
        self,
        user_idx: torch.Tensor,
        pos_item_idx: torch.Tensor,
        neg_item_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_vec = self.encode_user(
            user_idx,
            hist_items,
            hist_ratings,
            hist_timestamps,
            hist_mask,
            target_timestamp,
        )
        pos_vec = self.encode_item(pos_item_idx)
        neg_vec = self.encode_item(neg_item_idx)
        return (user_vec * pos_vec).sum(dim=-1), (user_vec * neg_vec).sum(dim=-1)

    def score_all_items_for_histories(self, user_vecs: torch.Tensor) -> torch.Tensor:
        item_idx = torch.arange(self.num_items, device=user_vecs.device)
        item_vecs = self.encode_item(item_idx)
        return user_vecs @ item_vecs.T


def build_model_and_loaders(cfg: ExperimentConfig, data: PreparedData):
    device = torch.device(cfg.train.device)
    warm_candidate_items = np.sort(data.train_df["item_idx"].astype(np.int64).unique())

    if cfg.model.model_type == "mf":
        pairwise_ds = PairwiseTrainDataset(data.train_df, data.seen_items, warm_candidate_items, cfg.train.positive_threshold)
        pairwise_loader = make_loader(pairwise_ds, cfg.train.batch_size, True, cfg)
        model = MFModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.mf,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        model = maybe_wrap_for_multi_gpu(model, cfg)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, pairwise_loader

    if cfg.model.model_type == "two_tower":
        seq_ds = SequencePairwiseDataset(
            data.train_df,
            warm_candidate_items,
            cfg.train.positive_threshold,
            cfg.train.max_history,
            cfg.train.min_history_for_sequence,
        )
        seq_loader = make_loader(seq_ds, cfg.train.batch_size, True, cfg, collate_fn=sequence_collate_fn)
        model = TwoTowerModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.two_tower,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        model = maybe_wrap_for_multi_gpu(model, cfg)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, seq_loader

    if cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
        edge_ds = ObservedEdgeDataset(data.train_df, data.rating2class)
        edge_loader = make_loader(edge_ds, cfg.train.relation_edge_batch_size, True, cfg)
        model_cls = SupportAwareGCMCModel if cfg.model.model_type == "support_gcmc" else RelationGCMCModel
        model = model_cls(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.graph,
            data.unique_ratings,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        graph_dict = build_relation_graph(
            data.train_df,
            data.num_users,
            data.num_items,
            data.unique_ratings,
            device,
            apply_time_decay=cfg.model.graph.relation_apply_time_decay,
            time_weight_fn_name=cfg.model.graph.time_weight_fn,
            gamma=cfg.model.graph.time_weight_gamma,
        )
        model.set_graph(graph_dict)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, edge_loader

    if cfg.model.model_type == "gcmc_to_two_tower":
        seq_ds = SequencePairwiseDataset(
            data.train_df,
            warm_candidate_items,
            cfg.train.positive_threshold,
            cfg.train.max_history,
            cfg.train.min_history_for_sequence,
        )
        seq_loader = make_loader(seq_ds, cfg.train.batch_size, True, cfg, collate_fn=sequence_collate_fn)

        graph_model = make_graph_model_for_hybrid(cfg, data, device)
        load_state_dict_if_available(
            graph_model,
            cfg.model.hybrid.graph_checkpoint_path,
            strict=False,
            require=cfg.model.hybrid.require_pretrained,
            name="graph_model",
        )
        model = GCMCToTwoTowerModel(graph_model, cfg.model.two_tower, cfg.model.hybrid).to(device)
        model.refresh_graph_cache()
        optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, seq_loader

    if cfg.model.model_type == "two_tower_to_gcmc":
        edge_ds = ObservedEdgeDataset(data.train_df, data.rating2class)
        edge_loader = make_loader(edge_ds, cfg.train.relation_edge_batch_size, True, cfg)

        tt_model = make_two_tower_model_for_hybrid(cfg, data, device)
        load_state_dict_if_available(
            tt_model,
            cfg.model.hybrid.two_tower_checkpoint_path,
            strict=False,
            require=cfg.model.hybrid.require_pretrained,
            name="two_tower_model",
        )
        model = TwoTowerToGCMCModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.graph,
            data.unique_ratings,
            tt_model,
            cfg.model.hybrid,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        graph_dict = build_relation_graph(
            data.train_df,
            data.num_users,
            data.num_items,
            data.unique_ratings,
            device,
            apply_time_decay=cfg.model.graph.relation_apply_time_decay,
            time_weight_fn_name=cfg.model.graph.time_weight_fn,
            gamma=cfg.model.graph.time_weight_gamma,
        )
        model.set_graph(graph_dict)
        optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, edge_loader

    if cfg.model.model_type == "weighted_tt_gcmc_ensemble":
        seq_ds = SequencePairwiseDataset(
            data.train_df,
            warm_candidate_items,
            cfg.train.positive_threshold,
            cfg.train.max_history,
            cfg.train.min_history_for_sequence,
        )
        seq_loader = make_loader(seq_ds, cfg.train.batch_size, True, cfg, collate_fn=sequence_collate_fn)

        tt_model = make_two_tower_model_for_hybrid(cfg, data, device)
        graph_model = make_graph_model_for_hybrid(cfg, data, device)
        load_state_dict_if_available(
            tt_model,
            cfg.model.hybrid.two_tower_checkpoint_path,
            strict=False,
            require=cfg.model.hybrid.require_pretrained,
            name="two_tower_model",
        )
        load_state_dict_if_available(
            graph_model,
            cfg.model.hybrid.graph_checkpoint_path,
            strict=False,
            require=cfg.model.hybrid.require_pretrained,
            name="graph_model",
        )
        model = WeightedTwoTowerGCMCEnsemble(tt_model, graph_model, cfg.model.hybrid).to(device)
        model.refresh_graph_cache()
        optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, seq_loader

    raise ValueError(f"Unknown model_type={cfg.model.model_type}")


def model_supports_eval_key(model_type: str, eval_key: str) -> bool:
    if eval_key.startswith("warm_"):
        return True
    if eval_key.startswith("cold_item_"):
        return model_type in {
            "mf", "two_tower", "relation_gcmc", "support_gcmc",
            "gcmc_to_two_tower", "two_tower_to_gcmc", "weighted_tt_gcmc_ensemble",
        }
    if eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_"):
        return model_type in {
            "two_tower", "support_gcmc",
            "gcmc_to_two_tower", "two_tower_to_gcmc", "weighted_tt_gcmc_ensemble",
        }
    return False


def train_one_epoch_pairwise(model, loader, optimizer, cfg: ExperimentConfig, logger: CometLogger, epoch: int) -> Dict[str, float]:
    device = torch.device(cfg.train.device)
    base_model = unwrap_model(model)
    if hasattr(base_model, "refresh_graph_cache"):
        base_model.refresh_graph_cache()

    model.train()
    # Keep frozen encoders in eval mode even while the wrapper is training.
    if hasattr(base_model, "graph_model") and getattr(base_model, "freeze_graph_encoder", False):
        base_model.graph_model.eval()
    if hasattr(base_model, "two_tower_model") and getattr(base_model, "freeze_two_tower_encoder", False):
        base_model.two_tower_model.eval()

    scaler = make_grad_scaler(cfg, device)
    total_loss = 0.0
    total_steps = 0
    total_examples = 0
    step_times = []
    grad_norms = []
    pbar = tqdm(loader, desc=f"Train epoch {epoch}", disable=not cfg.train.verbose)

    for step, batch in enumerate(pbar, start=1):
        step_start = time.perf_counter()
        batch = to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        with get_autocast_context(cfg, device):
            if hasattr(base_model, "score_batch"):
                pos_scores, neg_scores = base_model.score_batch(batch)
            elif cfg.model.model_type == "mf":
                pos_scores, neg_scores = model(batch["user_idx"], batch["pos_item_idx"], batch["neg_item_idx"])
            elif cfg.model.model_type == "two_tower":
                pos_scores, neg_scores = model(
                    batch["user_idx"],
                    batch["pos_item_idx"],
                    batch["neg_item_idx"],
                    batch["hist_items"],
                    batch["hist_ratings"],
                    batch["hist_timestamps"],
                    batch["hist_mask"],
                    batch["target_timestamp"],
                )
            else:
                raise ValueError(f"train_one_epoch_pairwise is not for model_type={cfg.model.model_type}")

            sample_w = rating_weight(batch["rating"], cfg.train.train_rating_weighting)
            loss = bpr_loss(pos_scores, neg_scores, sample_w)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = compute_grad_norm((p for p in model.parameters() if p.requires_grad))
        if cfg.train.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), cfg.train.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item())
        total_steps += 1
        total_examples += int(batch["user_idx"].shape[0])
        step_time = time.perf_counter() - step_start
        step_times.append(step_time)
        grad_norms.append(grad_norm)
        pbar.set_postfix(loss=f"{loss.item():.4f}", grad=f"{grad_norm:.3f}")

        log_payload = {
            "train/loss_step": float(loss.item()),
            "train/step_time_sec": float(step_time),
            "train/grad_norm_step": float(grad_norm),
            "train/examples_per_sec": float(batch["user_idx"].shape[0] / max(step_time, 1e-8)),
        }
        if hasattr(base_model, "alpha_user"):
            log_payload["train/alpha_user"] = float(base_model.alpha_user().detach().cpu())
        if hasattr(base_model, "alpha_item"):
            log_payload["train/alpha_item"] = float(base_model.alpha_item().detach().cpu())

        if step % cfg.train.log_every_n_steps == 0:
            logger.log_metrics(
                log_payload,
                step=(epoch - 1) * len(loader) + step,
                epoch=epoch,
            )

    out = {
        "train_loss": float(total_loss / max(total_steps, 1)),
        "train_step_time_mean_sec": float(np.mean(step_times)) if step_times else 0.0,
        "train_grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else 0.0,
        "train_grad_norm_max": float(np.max(grad_norms)) if grad_norms else 0.0,
        "train_examples_per_sec": float(total_examples / max(np.sum(step_times), 1e-8)) if step_times else 0.0,
        "train_steps": int(total_steps),
    }
    if hasattr(base_model, "alpha_user"):
        out["alpha_user"] = float(base_model.alpha_user().detach().cpu())
    if hasattr(base_model, "alpha_item"):
        out["alpha_item"] = float(base_model.alpha_item().detach().cpu())
    return out


def compute_eval_loss(model, data: PreparedData, cfg: ExperimentConfig, eval_key: str) -> Dict[str, float]:
    if not cfg.eval.compute_loss:
        return {}
    if eval_key not in data.eval_frames or not model_supports_eval_key(cfg.model.model_type, eval_key):
        return {}

    eval_df = data.eval_frames[eval_key]
    if eval_df.empty:
        return {}

    if cfg.eval.loss_max_rows is not None and len(eval_df) > cfg.eval.loss_max_rows:
        eval_df = eval_df.sample(cfg.eval.loss_max_rows, random_state=cfg.train.seed).reset_index(drop=True)

    device = torch.device(cfg.train.device)
    base_model = unwrap_model(model)
    base_model.eval()
    if hasattr(base_model, "refresh_graph_cache"):
        base_model.refresh_graph_cache()

    user_histories = data.eval_histories.get(eval_key, data.user_histories)
    base_seen_items = data.eval_seen_items.get(eval_key, data.seen_items)
    observed_eval_items = defaultdict(set)
    for row in eval_df[["user_idx", "item_idx"]].itertuples(index=False):
        observed_eval_items[int(row.user_idx)].add(int(row.item_idx))
    exclusion_by_user = {
        int(u): set(base_seen_items.get(int(u), set())) | observed_eval_items.get(int(u), set())
        for u in set(base_seen_items.keys()) | set(observed_eval_items.keys())
    }

    if cfg.model.model_type in PAIRWISE_MODEL_TYPES:
        pos_df = eval_df.loc[eval_df["rating"] >= cfg.train.positive_threshold, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if pos_df.empty:
            return {"loss": 0.0, "loss_rows": 0}
        rng = np.random.default_rng(_stable_seed_from_text(eval_key, cfg.train.seed))
        total_loss = 0.0
        total_rows = 0
        batch_size = max(1, cfg.eval.loss_batch_size)

        with torch.no_grad():
            for start in range(0, len(pos_df), batch_size):
                batch_df = pos_df.iloc[start: start + batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                pos_item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                ratings = batch_df["rating"].to_numpy(dtype=np.float32)
                timestamps = batch_df["timestamp"].to_numpy(dtype=np.int64)
                neg_item_idx = np.array([
                    _sample_negative_item(exclusion_by_user.get(int(u), set()), data.num_items, rng)
                    for u in user_idx
                ], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                pos_t = torch.tensor(pos_item_idx, dtype=torch.long, device=device)
                neg_t = torch.tensor(neg_item_idx, dtype=torch.long, device=device)
                rating_t = torch.tensor(ratings, dtype=torch.float32, device=device)

                with get_autocast_context(cfg, device):
                    if cfg.model.model_type == "mf":
                        pos_scores = base_model.score(user_t, pos_t)
                        neg_scores = base_model.score(user_t, neg_t)
                    else:
                        hist_batch = build_eval_history_tensors_for_rows(
                            user_indices=user_idx.tolist(),
                            target_timestamps=timestamps.tolist(),
                            user_histories=user_histories,
                            max_history=cfg.train.max_history,
                            device=device,
                        )
                        user_vec = base_model.encode_user(user_t, **hist_batch)
                        pos_scores = (user_vec * base_model.encode_item(pos_t)).sum(dim=-1)
                        neg_scores = (user_vec * base_model.encode_item(neg_t)).sum(dim=-1)

                    sample_w = rating_weight(rating_t, cfg.train.train_rating_weighting)
                    loss = bpr_loss(pos_scores, neg_scores, sample_w)

                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows

        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}

    if cfg.model.model_type in GRAPH_MODEL_TYPES:
        valid_mask = eval_df["rating"].isin(data.rating2class.keys())
        eval_obs = eval_df.loc[valid_mask, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if eval_obs.empty:
            return {"loss": 0.0, "loss_rows": 0}

        graph_batch_size = max(1, cfg.eval.loss_batch_size)
        total_loss = 0.0
        total_rows = 0
        with torch.no_grad():
            with get_autocast_context(cfg, device):
                graph_user_z, graph_item_z = base_model.encode_full_graph()
            for start in range(0, len(eval_obs), graph_batch_size):
                batch_df = eval_obs.iloc[start: start + graph_batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                timestamps = batch_df["timestamp"].to_numpy(dtype=np.int64)
                rating_class = np.array([data.rating2class[float(r)] for r in batch_df["rating"].tolist()], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                item_t = torch.tensor(item_idx, dtype=torch.long, device=device)
                class_t = torch.tensor(rating_class, dtype=torch.long, device=device)

                with get_autocast_context(cfg, device):
                    if hasattr(base_model, "encode_support_users") and (eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_")):
                        hist_batch = build_eval_history_tensors_for_rows(
                            user_indices=user_idx.tolist(),
                            target_timestamps=timestamps.tolist(),
                            user_histories=user_histories,
                            max_history=cfg.train.max_history,
                            device=device,
                        )
                        user_vec = base_model.encode_support_users(**hist_batch, item_vectors=graph_item_z)
                    else:
                        user_vec = graph_user_z[user_t]
                    logits = base_model.decode(user_vec, graph_item_z[item_t])
                    loss = F.cross_entropy(logits, class_t, reduction="mean")

                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows
        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}

    return {}


@torch.inference_mode()
def evaluate_model(model, data: PreparedData, cfg: ExperimentConfig, eval_key: str = "warm_val") -> Dict[str, float]:
    if eval_key not in data.eval_frames:
        return {"skipped": 1.0}
    if not model_supports_eval_key(cfg.model.model_type, eval_key):
        return {"skipped": 1.0}

    base_model = unwrap_model(model)
    device = torch.device(cfg.train.device)
    base_model.eval()
    if hasattr(base_model, "refresh_graph_cache"):
        base_model.refresh_graph_cache()

    eval_df = data.eval_frames[eval_key]
    if eval_df.empty:
        return {"users_scored": 0, "time_sec": 0.0, "empty": 1.0}

    gain_types = get_metric_gain_types(cfg.eval)
    targets_by_gain = {gain_type: build_eval_targets(eval_df, cfg.eval, gain_type) for gain_type in gain_types}
    users = sorted(set().union(*[set(v.keys()) for v in targets_by_gain.values()]))
    if not users:
        return {}

    metric_rows_by_gain: Dict[str, List[Dict[str, float]]] = {gain_type: [] for gain_type in gain_types}
    user_histories = data.eval_histories.get(eval_key, data.user_histories)
    seen_items_by_user = data.eval_seen_items.get(eval_key, data.seen_items)
    t0 = time.perf_counter()

    graph_user_z = None
    graph_item_z = None
    if cfg.model.model_type in GRAPH_MODEL_TYPES:
        with get_autocast_context(cfg, device):
            graph_user_z, graph_item_z = base_model.encode_full_graph()

    for start_idx in tqdm(range(0, len(users), cfg.eval.user_batch_size), desc=f"Eval {eval_key}"):
        batch_users = users[start_idx: start_idx + cfg.eval.user_batch_size]
        user_tensor = torch.tensor(batch_users, dtype=torch.long, device=device)

        with get_autocast_context(cfg, device):
            if cfg.model.model_type == "mf":
                score_matrix = base_model.score_all_items(user_tensor)

            elif cfg.model.model_type in {"two_tower", "gcmc_to_two_tower", "weighted_tt_gcmc_ensemble"}:
                hist_batch = build_eval_history_tensors(batch_users, user_histories, cfg.train.max_history, device)
                user_vecs = base_model.encode_user(user_tensor, **hist_batch)
                score_matrix = base_model.score_all_items_for_histories(user_vecs)

            elif cfg.model.model_type == "relation_gcmc":
                score_matrix = base_model.score_user_vectors_all_items(
                    graph_user_z[user_tensor],
                    graph_item_z,
                    positive_threshold=cfg.eval.positive_threshold,
                )

            elif cfg.model.model_type in {"support_gcmc", "two_tower_to_gcmc"}:
                if eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_"):
                    hist_batch = build_eval_history_tensors(batch_users, user_histories, cfg.train.max_history, device)
                    user_vecs = base_model.encode_support_users(**hist_batch, item_vectors=graph_item_z)
                else:
                    user_vecs = graph_user_z[user_tensor]
                score_matrix = base_model.score_user_vectors_all_items(
                    user_vecs,
                    graph_item_z,
                    positive_threshold=cfg.eval.positive_threshold,
                )

            else:
                raise ValueError(f"Unsupported model_type={cfg.model.model_type}")

        score_matrix = score_matrix.detach().float().cpu().numpy().astype(np.float32)
        max_k = max(cfg.eval.ks)

        for row_idx, user_idx in enumerate(batch_users):
            scores = score_matrix[row_idx]
            if cfg.eval.exclude_seen_items:
                seen = seen_items_by_user.get(int(user_idx), set())
                if seen:
                    seen_idx = np.fromiter(seen, dtype=np.int64)
                    valid_seen = seen_idx[(seen_idx >= 0) & (seen_idx < len(scores))]
                    if valid_seen.size > 0:
                        scores[valid_seen] = -1e9

            kth = min(max_k, len(scores) - 1)
            topk_idx = np.argpartition(-scores, kth=kth)[:max_k]
            ranked_items = topk_idx[np.argsort(-scores[topk_idx])]

            for gain_type in gain_types:
                target_gain = targets_by_gain[gain_type].get(int(user_idx), {})
                if not target_gain and cfg.eval.skip_users_with_no_relevant_targets:
                    continue
                metric_names = get_eval_metric_names(cfg.eval, gain_type)
                row_metrics = evaluate_ranked_items(
                    ranked_items=ranked_items,
                    target_gain_by_item=target_gain,
                    ks=cfg.eval.ks,
                    metric_names=metric_names,
                )
                metric_rows_by_gain[gain_type].append(row_metrics)

    elapsed = time.perf_counter() - t0
    out = {"users_scored": len(users), "time_sec": float(elapsed)}
    out.update(compute_eval_loss(base_model, data, cfg, eval_key))

    for gain_type, rows in metric_rows_by_gain.items():
        if not rows:
            continue
        df = pd.DataFrame(rows)
        for col in df.columns:
            out[f"{gain_type}/{col}"] = float(df[col].mean())

    if hasattr(base_model, "alpha_user"):
        out["alpha_user"] = float(base_model.alpha_user().detach().cpu())
    if hasattr(base_model, "alpha_item"):
        out["alpha_item"] = float(base_model.alpha_item().detach().cpu())

    return out


def run_training(cfg: ExperimentConfig):
    set_seed(cfg.train.seed)
    configure_runtime(cfg)
    data = load_prepared_data(cfg)
    model, optimizer, train_loader = build_model_and_loaders(cfg, data)
    logger = CometLogger(cfg)
    runtime_info = get_gpu_info()
    logger.log_metrics({
        "runtime/num_gpus": runtime_info["num_gpus"],
        "runtime/use_amp": float(cfg.train.use_amp),
        "runtime/use_multi_gpu": float(cfg.train.use_multi_gpu),
    })
    if runtime_info["gpu_names"]:
        logger.log_text("gpu_names", ", ".join(runtime_info["gpu_names"]))

    save_json(dc_to_dict(cfg), data.output_dir / f"config_{cfg.model.model_type}.json")

    best_metric = -1e18
    best_epoch = -1
    best_metrics = None
    patience_counter = 0
    history = []

    val_eval_keys = get_eval_keys(cfg, "val")
    test_eval_keys = get_eval_keys(cfg, "test")

    for epoch in range(1, cfg.train.epochs + 1):
        print("=" * 80)
        print(f"Epoch {epoch}/{cfg.train.epochs} | model_type={cfg.model.model_type}")
        epoch_start = time.perf_counter()

        if cfg.model.model_type in PAIRWISE_MODEL_TYPES:
            train_metrics = train_one_epoch_pairwise(model, train_loader, optimizer, cfg, logger, epoch)
        elif cfg.model.model_type in GRAPH_MODEL_TYPES:
            train_metrics = train_one_epoch_relation_gcmc(model, train_loader, optimizer, cfg, data, logger, epoch)
        else:
            raise ValueError(cfg.model.model_type)

        val_by_scenario: Dict[str, Dict[str, float]] = {}
        flat_val_metrics: Dict[str, float] = {}
        for eval_key in val_eval_keys:
            scenario_metrics = evaluate_model(model, data, cfg, eval_key=eval_key)
            val_by_scenario[eval_key] = scenario_metrics
            flat_val_metrics.update({f"val/{eval_key}/{k}": v for k, v in scenario_metrics.items()})

        epoch_time = time.perf_counter() - epoch_start
        epoch_metrics = {**train_metrics, **flat_val_metrics, "epoch_time_sec": float(epoch_time)}
        logger.log_metrics(epoch_metrics, epoch=epoch)
        history.append({"epoch": epoch, **epoch_metrics})

        print("TRAIN | " + " | ".join(f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}" for k, v in train_metrics.items()))
        for eval_key, scenario_metrics in val_by_scenario.items():
            printable = " | ".join(f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}" for k, v in scenario_metrics.items())
            print(f"VAL[{eval_key}] | {printable}")
        print(f"TIME  | epoch_time_sec={epoch_time:.3f}")

        current_metric = select_main_metric(val_by_scenario.get("warm_val", {}), cfg.eval)
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_metrics = val_by_scenario.get("warm_val", {})
            patience_counter = 0
            save_checkpoint(model, data.output_dir, f"best_{cfg.model.model_type}.pt")
        else:
            patience_counter += 1
            print(f"[early-stop] no improvement for {patience_counter} epoch(s)")
            if patience_counter >= cfg.train.patience:
                print("[early-stop] stopping training")
                break

    print("=" * 80)
    print(f"Best epoch={best_epoch} | main_metric={best_metric:.6f}")
    hist_df = pd.DataFrame(history)
    hist_path = data.output_dir / f"history_{cfg.model.model_type}.csv"
    hist_df.to_csv(hist_path, index=False)
    print(f"Saved training history to: {hist_path}")

    best_ckpt = data.output_dir / f"best_{cfg.model.model_type}.pt"
    if best_ckpt.exists():
        unwrap_model(model).load_state_dict(torch.load(best_ckpt, map_location=cfg.train.device))

    test_by_scenario: Dict[str, Dict[str, float]] = {}
    flat_test_metrics: Dict[str, float] = {}
    for eval_key in test_eval_keys:
        scenario_metrics = evaluate_model(model, data, cfg, eval_key=eval_key)
        test_by_scenario[eval_key] = scenario_metrics
        flat_test_metrics.update({f"test/{eval_key}/{k}": v for k, v in scenario_metrics.items()})
    logger.log_metrics(flat_test_metrics)
    logger.end()

    return model, data, hist_df, best_metrics, test_by_scenario


In [ ]:
import os

# Optional Comet setup.
# Put COMET_API_KEY into Kaggle Secrets or set it in the environment before running.
# Do not hard-code API keys in the notebook.
os.environ.setdefault("COMET_PROJECT_NAME", "explicit-movie-ranking")


## Common config factory

In [ ]:

def make_common_cfg(model_type: str, experiment_name: str) -> ExperimentConfig:
    cfg = ExperimentConfig()
    cfg.paths.dataset_dir = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
    cfg.paths.output_dir = "/kaggle/working/hybrid_two_tower_gcmc"
    cfg.model.model_type = model_type

    # Keep architecture dimensions equal between Two-Tower and GCMC.
    cfg.model.two_tower.embedding_dim = 128
    cfg.model.two_tower.hidden_dim = 256
    cfg.model.two_tower.dropout = 0.10
    cfg.model.two_tower.use_user_id_embedding = True
    cfg.model.two_tower.use_rating_in_history = True
    cfg.model.two_tower.use_time_in_history = True
    cfg.model.two_tower.rating_weight_fn = "none"      # like your successful run
    cfg.model.two_tower.time_weight_fn = "exp"
    cfg.model.two_tower.time_weight_gamma = 1e-8
    cfg.model.two_tower.sasrec_num_layers = 2
    cfg.model.two_tower.sasrec_num_heads = 4
    cfg.model.two_tower.sasrec_ffn_dim = 256
    cfg.model.two_tower.sasrec_attention_dropout = 0.10
    cfg.model.two_tower.sasrec_max_seq_len = 512
    cfg.model.two_tower.sasrec_use_causal_mask = True

    cfg.model.graph.embedding_dim = 128
    cfg.model.graph.hidden_dim = 128
    cfg.model.graph.num_layers = 2
    cfg.model.graph.dropout = 0.10
    cfg.model.graph.rating_weight_fn = "linear_norm"
    cfg.model.graph.relation_score_mode = "expected_rating"
    cfg.model.graph.relation_apply_time_decay = False
    cfg.model.graph.support_history_pooling = "weighted_mean"
    cfg.model.graph.support_use_rating = True
    cfg.model.graph.support_use_time = True
    cfg.model.graph.support_time_weight_fn = "exp"
    cfg.model.graph.support_time_weight_gamma = 1e-8
    cfg.model.graph.support_alignment_weight = 0.2

    cfg.features.use_item_features = True
    cfg.features.item_feature_mode = "add"
    cfg.features.scaler_kind = "standard"
    cfg.features.fit_scaler_on = "warm"
    cfg.features.feature_dropout = 0.10
    cfg.features.feature_hidden_dim = 256

    cfg.train.seed = 42
    cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"
    cfg.train.epochs = 10
    cfg.train.lr = 3e-4
    cfg.train.weight_decay = 1e-5
    cfg.train.batch_size = 256
    cfg.train.num_workers = 4
    cfg.train.patience = 4
    cfg.train.grad_clip_norm = 5.0
    cfg.train.positive_threshold = 4.0
    cfg.train.train_rating_weighting = "none"
    cfg.train.max_history = 50
    cfg.train.min_history_for_sequence = 1
    cfg.train.relation_edge_batch_size = 131072
    cfg.train.graph_loss_batches_per_epoch_log = 10
    cfg.train.use_amp = True
    cfg.train.tf32 = True

    # Hybrids keep frozen pretrained encoders by default; DataParallel is disabled for simpler caches.
    cfg.train.use_multi_gpu = False

    # cfg.eval.eval_scenarios = ("warm", "cold_item")
    # For full report, switch to:
    cfg.eval.eval_scenarios = ("warm", "cold_user", "cold_item", "both_cold")
    cfg.eval.evaluate_cold_on_val = True
    cfg.eval.evaluate_cold_on_test = True
    cfg.eval.cold_user_use_support_all = True
    cfg.eval.main_metric_gain_type = "centered_3"
    cfg.eval.main_metric_name = "ndcg"
    cfg.eval.main_metric_k = 10
    cfg.eval.user_batch_size = 256
    cfg.eval.loss_batch_size = 4096

    cfg.comet.enabled = True
    cfg.comet.project_name = "explicit-movie-ranking"
    cfg.comet.experiment_name = experiment_name
    cfg.comet.tags = ["hybrid", model_type]

    return cfg


ALL_SCENARIOS = ("warm", "cold_user", "cold_item", "both_cold")

def enable_full_warm_cold_eval(cfg):
    cfg.eval.eval_scenarios = ALL_SCENARIOS
    cfg.eval.evaluate_cold_on_val = True
    cfg.eval.evaluate_cold_on_test = True
    cfg.eval.cold_user_use_support_all = True

    cfg.eval.ks = (10, 20, 50)
    cfg.eval.exclude_seen_items = True
    cfg.eval.skip_users_with_no_relevant_targets = True
    cfg.eval.positive_threshold = 4.0

    cfg.eval.metric_gain_types = ("binary_ge_4", "centered_3", "power")
    cfg.eval.binary_metric_names = ("precision", "recall", "hitrate", "mrr", "map", "ndcg")
    cfg.eval.graded_metric_names = ("ndcg",)

    cfg.eval.neutral_rating = 3.0
    cfg.eval.power_beta = 0.2
    cfg.eval.power_gamma = 2.0

    cfg.eval.main_metric_gain_type = "centered_3"
    cfg.eval.main_metric_name = "ndcg"
    cfg.eval.main_metric_k = 10

    cfg.eval.user_batch_size = 256
    cfg.eval.loss_batch_size = 4096
    cfg.eval.compute_loss = True

    return cfg


# Expected default checkpoint locations if you train/load base models in the same output directory.
DEFAULT_TWO_TOWER_CKPT = "/kaggle/working/hybrid_two_tower_gcmc/best_two_tower.pt"
DEFAULT_GRAPH_CKPT = "/kaggle/working/hybrid_two_tower_gcmc/best_support_gcmc.pt"
# If you already have checkpoints from another output_dir, put their paths in the configs below.


## Updated patch: init-only features + graph CE/BPR training

Эта секция делает два важных изменения:

1. `cfg.features.item_feature_mode = "init_only"` теперь означает, что item features используются только для начальной инициализации item embedding через SVD-проекцию, а дальше обучается обычный embedding.
2. Для `relation_gcmc` / `support_gcmc` графовый loss становится комбинированным: `CE + aux_bpr_weight * BPR`. CE обучает предсказание explicit rating-класса, а BPR добавляет ranking-сигнал для `rating >= positive_threshold`.


In [ ]:
# ============================================================
# Patch 1: FeatureAwareItemEncoder with item_feature_mode="init_only"
# ============================================================
# init_only:
# - item features are projected to embedding_dim once during initialization;
# - forward() then uses only trainable item_id_emb;
# - cold items are NOT zero-masked, because their initial vectors come from features.

class FeatureAwareItemEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        feature_tensor: torch.Tensor,
        embedding_dim: int,
        feature_cfg: FeatureConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.embedding_dim = int(embedding_dim)
        self.feature_cfg = feature_cfg

        self.item_id_emb = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.item_id_emb.weight, std=0.02)

        self.register_buffer("item_features", feature_tensor)
        if trainable_item_mask is None:
            trainable_item_mask = torch.ones(num_items, dtype=torch.float32)
        self.register_buffer("trainable_item_mask", trainable_item_mask.float().view(-1, 1))

        feat_dim = int(feature_tensor.shape[1]) if feature_tensor is not None else 0
        self.use_features = feature_tensor is not None and feature_cfg.use_item_features and feat_dim > 0
        self.init_only_mode = bool(self.use_features and feature_cfg.item_feature_mode == "init_only")
        self.feature_dropout = nn.Dropout(feature_cfg.feature_dropout)

        if self.use_features and not self.init_only_mode:
            self.feature_proj = nn.Linear(feat_dim, embedding_dim)
            nn.init.xavier_uniform_(self.feature_proj.weight)
            nn.init.zeros_(self.feature_proj.bias)
            if feature_cfg.item_feature_mode == "concat_mlp":
                self.concat_mlp = nn.Sequential(
                    nn.Linear(embedding_dim * 2, feature_cfg.feature_hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(feature_cfg.feature_dropout),
                    nn.Linear(feature_cfg.feature_hidden_dim, embedding_dim),
                )
            else:
                self.concat_mlp = None
        else:
            self.feature_proj = None
            self.concat_mlp = None

        if self.init_only_mode:
            init_vectors = self._build_init_only_vectors(feature_tensor)
            with torch.no_grad():
                self.item_id_emb.weight.copy_(init_vectors)

    def _build_init_only_vectors(self, feature_tensor: torch.Tensor) -> torch.Tensor:
        feat_np = feature_tensor.detach().cpu().numpy().astype(np.float32, copy=False)
        if feat_np.ndim != 2 or feat_np.shape[0] == 0:
            return self.item_id_emb.weight.detach().clone()

        if feat_np.shape[1] == self.embedding_dim:
            init_np = feat_np
        else:
            max_components = min(feat_np.shape[1], max(1, feat_np.shape[0] - 1))
            n_components = max(1, min(self.embedding_dim, max_components))
            svd = TruncatedSVD(
                n_components=n_components,
                n_iter=int(getattr(self.feature_cfg, "init_only_svd_n_iter", 7)),
                random_state=int(getattr(self.feature_cfg, "init_only_random_state", 42)),
            )
            init_np = svd.fit_transform(feat_np).astype(np.float32)
            if n_components < self.embedding_dim:
                pad = np.zeros((init_np.shape[0], self.embedding_dim - n_components), dtype=np.float32)
                init_np = np.concatenate([init_np, pad], axis=1)

        init_tensor = torch.tensor(init_np, dtype=torch.float32)
        init_std = float(init_tensor.std().item()) if init_tensor.numel() > 0 else 0.0
        if init_std > 1e-8:
            init_tensor = init_tensor / init_std * 0.02
        return init_tensor

    def forward(self, item_idx: torch.Tensor) -> torch.Tensor:
        raw_item_id_vec = self.item_id_emb(item_idx)

        if self.init_only_mode:
            return raw_item_id_vec

        item_id_vec = raw_item_id_vec * self.trainable_item_mask[item_idx]
        if not self.use_features:
            return item_id_vec

        feat_vec = self.feature_proj(self.feature_dropout(self.item_features[item_idx]))
        mode = self.feature_cfg.item_feature_mode
        if mode == "none":
            return item_id_vec
        if mode == "add":
            return item_id_vec + feat_vec
        if mode == "project_only":
            return feat_vec
        if mode == "concat_mlp":
            return self.concat_mlp(torch.cat([item_id_vec, feat_vec], dim=-1))
        raise ValueError(f"Unknown item_feature_mode={mode}")

    def all_item_vectors(self) -> torch.Tensor:
        idx = torch.arange(self.num_items, device=self.item_id_emb.weight.device)
        return self.forward(idx)


# ============================================================
# Patch 2: Graph training with CE + auxiliary BPR
# ============================================================

def _sample_graph_neg_items_for_batch(
    user_idx: torch.Tensor,
    data: PreparedData,
    device: torch.device,
) -> torch.Tensor:
    # Training graph is warm-only, so negatives are sampled from warm train items.
    candidate_items = np.sort(data.train_df["item_idx"].astype(np.int64).unique())
    if candidate_items.size == 0:
        candidate_items = np.arange(data.num_items, dtype=np.int64)

    neg_items = []
    users_cpu = user_idx.detach().cpu().numpy().astype(np.int64)
    for u in users_cpu:
        seen = data.seen_items.get(int(u), set())
        neg = int(candidate_items[random.randrange(len(candidate_items))])
        tries = 0
        while neg in seen and tries < 64:
            neg = int(candidate_items[random.randrange(len(candidate_items))])
            tries += 1
        if neg in seen:
            # Rare fallback for users who have seen nearly the whole candidate set.
            unseen = np.setdiff1d(candidate_items, np.fromiter(seen, dtype=np.int64), assume_unique=False)
            if unseen.size > 0:
                neg = int(unseen[random.randrange(len(unseen))])
        neg_items.append(neg)

    return torch.tensor(neg_items, dtype=torch.long, device=device)


def train_one_epoch_relation_gcmc(
    model,
    edge_loader,
    optimizer,
    cfg,
    data,
    logger,
    epoch: int,
):
    device = torch.device(cfg.train.device)
    model.train()

    scaler = make_grad_scaler(cfg, device)
    optimizer.zero_grad(set_to_none=True)

    forward_start = time.perf_counter()
    with get_autocast_context(cfg, device):
        user_z, item_z = model.encode_full_graph()
    forward_time = time.perf_counter() - forward_start

    total_loss = 0.0
    total_ce = 0.0
    total_bpr = 0.0
    total_edges = 0
    total_bpr_edges = 0
    num_batches = 0
    align_loss_value = 0.0

    dataset_size = len(edge_loader.dataset)
    bpr_weight = float(getattr(cfg.model.graph, "aux_bpr_weight", 0.0))
    bpr_pos_threshold = float(getattr(cfg.model.graph, "aux_bpr_positive_threshold", cfg.train.positive_threshold))
    support_align_enabled = (
        hasattr(unwrap_model(model), "encode_support_users")
        and cfg.model.graph.support_alignment_weight > 0
    )

    pbar = tqdm(edge_loader, desc=f"Graph CE+BPR epoch {epoch}", disable=not cfg.train.verbose)
    backward_start = time.perf_counter()

    for step, batch in enumerate(pbar, start=1):
        batch = to_device(batch, device)

        with get_autocast_context(cfg, device):
            user_vec = user_z[batch["user_idx"]]
            pos_item_vec = item_z[batch["item_idx"]]
            logits = model.decode(user_vec, pos_item_vec)
            ce = F.cross_entropy(logits, batch["rating_class"], reduction="mean")

            if bpr_weight > 0:
                pos_mask = batch["rating"] >= bpr_pos_threshold
                if bool(pos_mask.any()):
                    neg_item_idx = _sample_graph_neg_items_for_batch(batch["user_idx"][pos_mask], data, device)
                    pos_logits = logits[pos_mask]
                    neg_logits = model.decode(user_z[batch["user_idx"][pos_mask]], item_z[neg_item_idx])

                    pos_scores = model.score_from_logits(pos_logits, positive_threshold=cfg.train.positive_threshold)
                    neg_scores = model.score_from_logits(neg_logits, positive_threshold=cfg.train.positive_threshold)
                    sample_w = rating_weight(batch["rating"][pos_mask], cfg.train.train_rating_weighting)
                    bpr = bpr_loss(pos_scores, neg_scores, sample_w)
                    bpr_edges = int(pos_mask.sum().item())
                else:
                    bpr = ce.new_zeros(())
                    bpr_edges = 0
            else:
                bpr = ce.new_zeros(())
                bpr_edges = 0

            loss = ce + bpr_weight * bpr

        batch_size = int(batch["user_idx"].shape[0])
        batch_weight = batch_size / max(dataset_size, 1)
        scaled_loss = loss * batch_weight

        retain = (step < len(edge_loader)) or support_align_enabled
        scaler.scale(scaled_loss).backward(retain_graph=retain)

        total_loss += float(loss.item()) * batch_size
        total_ce += float(ce.item()) * batch_size
        total_bpr += float(bpr.item()) * max(bpr_edges, 1)
        total_edges += batch_size
        total_bpr_edges += bpr_edges
        num_batches += 1

        if step % max(1, cfg.train.graph_loss_batches_per_epoch_log) == 0:
            logger.log_metrics(
                {
                    "train/relation_batch_total": float(loss.item()),
                    "train/relation_batch_ce": float(ce.item()),
                    "train/relation_batch_bpr": float(bpr.item()),
                    "train/relation_batch_bpr_edges": int(bpr_edges),
                    "train/relation_aux_bpr_weight": float(bpr_weight),
                },
                step=(epoch - 1) * len(edge_loader) + step,
                epoch=epoch,
            )

        pbar.set_postfix(total=f"{loss.item():.4f}", ce=f"{ce.item():.4f}", bpr=f"{bpr.item():.4f}")

        del batch, logits, loss, ce, bpr, scaled_loss

    base_graph_model = unwrap_model(model)

    if support_align_enabled:
        with get_autocast_context(cfg, device):
            align_loss = compute_support_alignment_loss(
                base_graph_model,
                data,
                cfg,
                device,
                user_z,
                item_z,
            )
        align_loss_value = float(align_loss.item())
        scaled_align_loss = cfg.model.graph.support_alignment_weight * align_loss
        scaler.scale(scaled_align_loss).backward()
        del align_loss, scaled_align_loss

    scaler.unscale_(optimizer)
    grad_norm = compute_grad_norm((p for p in model.parameters() if p.requires_grad))
    if cfg.train.grad_clip_norm is not None:
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), cfg.train.grad_clip_norm)

    scaler.step(optimizer)
    scaler.update()

    backward_time = time.perf_counter() - backward_start

    del user_z, item_z
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "train_loss": float(total_loss / max(total_edges, 1)),
        "train_ce_loss": float(total_ce / max(total_edges, 1)),
        "train_bpr_loss": float(total_bpr / max(total_bpr_edges, 1)) if total_bpr_edges > 0 else 0.0,
        "train_bpr_edges": int(total_bpr_edges),
        "graph_aux_bpr_weight": float(bpr_weight),
        "graph_forward_time_sec": float(forward_time),
        "graph_backward_time_sec": float(backward_time),
        "train_grad_norm_mean": float(grad_norm),
        "train_grad_norm_max": float(grad_norm),
        "train_edges_per_sec": float(total_edges / max(forward_time + backward_time, 1e-8)),
        "support_alignment_loss": float(align_loss_value),
        "train_batches": int(num_batches),
    }


# Compatibility properties needed by some hybrids.
if not hasattr(RelationGCMCModel, "num_users"):
    RelationGCMCModel.num_users = property(lambda self: self.user_emb.num_embeddings)
if not hasattr(RelationGCMCModel, "num_items"):
    RelationGCMCModel.num_items = property(lambda self: self.item_encoder.num_items)

print("Patched: init_only item features + graph CE/BPR training")


## Sequential experiment: Two-Tower → Support-GCMC CE+BPR → weighted ensemble


In [ ]:
# ============================================================
# Common paths and config helpers for the sequential experiment
# ============================================================

DATASET_DIR = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
OUT_DIR = "/kaggle/working/hybrid_two_tower_gcmc"

TT_CKPT = f"{OUT_DIR}/best_two_tower.pt"
GCMC_CKPT = f"{OUT_DIR}/best_support_gcmc.pt"
ENSEMBLE_CKPT = f"{OUT_DIR}/best_weighted_tt_gcmc_ensemble.pt"


def apply_init_only_features(cfg):
    cfg.features.use_item_features = True
    cfg.features.item_feature_mode = "init_only"
    cfg.features.scaler_kind = "standard"
    cfg.features.fit_scaler_on = "warm"
    cfg.features.feature_dropout = 0.10
    cfg.features.feature_hidden_dim = 256
    cfg.features.init_only_svd_n_iter = 7
    cfg.features.init_only_random_state = 42
    return cfg


def apply_full_eval(cfg):
    cfg = enable_full_warm_cold_eval(cfg)
    cfg.eval.user_batch_size = 128
    cfg.eval.loss_batch_size = 2048
    cfg.eval.compute_loss = True
    return cfg


def apply_common_train_runtime(cfg):
    cfg.train.seed = 42
    cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"
    cfg.train.tf32 = True
    cfg.train.num_workers = 4
    cfg.train.persistent_workers = True
    cfg.train.prefetch_factor = 2
    cfg.train.verbose = True
    cfg.train.positive_threshold = 4.0
    return cfg


print("DATASET_DIR:", DATASET_DIR)
print("OUT_DIR:", OUT_DIR)
print("TT_CKPT:", TT_CKPT)
print("GCMC_CKPT:", GCMC_CKPT)


In [ ]:
# ============================================================
# PATCH: safe unfrozen Weighted Two-Tower + GCMC ensemble
# ============================================================
# Исправляет:
# RuntimeError: Trying to backward through the graph a second time
#
# Причина:
# unfrozen GCMC graph-cache содержит autograd graph.
# Его нельзя переиспользовать на нескольких batch-ах подряд.
#
# Что делает patch:
# 1) не строит graph-cache при создании ensemble, если GCMC не заморожен;
# 2) для каждого train batch пересчитывает graph-cache заново;
# 3) после backward/optimizer.step сразу очищает graph-cache.

import time
import numpy as np
import torch
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Patch refresh_graph_cache
# ------------------------------------------------------------

def _patched_refresh_graph_cache_for_unfrozen_ensemble(self) -> None:
    if not hasattr(self, "graph_model"):
        return

    # Если graph frozen или модель в eval, cache должен быть detached.
    # Если train + graph unfrozen, cache нужен с autograd,
    # но только для одного batch-а.
    should_detach = bool(getattr(self, "freeze_graph_encoder", True)) or (not self.training)

    if should_detach:
        was_training = self.graph_model.training
        self.graph_model.eval()
        with torch.no_grad():
            user_z, item_z = self.graph_model.encode_full_graph()
        if was_training and self.training:
            self.graph_model.train()

        user_z = user_z.detach()
        item_z = item_z.detach()
    else:
        user_z, item_z = self.graph_model.encode_full_graph()

    self._graph_user_z = user_z
    self._graph_item_z = item_z


GraphCacheMixin.refresh_graph_cache = _patched_refresh_graph_cache_for_unfrozen_ensemble


def _clear_graph_cache_if_present(model):
    base_model = unwrap_model(model)

    if hasattr(base_model, "_graph_user_z"):
        base_model._graph_user_z = None
    if hasattr(base_model, "_graph_item_z"):
        base_model._graph_item_z = None

    # В Kaggle лучше чистить агрессивно, чтобы не копился старый graph.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 2. Patch build_model_and_loaders for weighted ensemble
# ------------------------------------------------------------

if "_ORIG_BUILD_MODEL_AND_LOADERS_BEFORE_UNFROZEN_ENSEMBLE_PATCH" not in globals():
    _ORIG_BUILD_MODEL_AND_LOADERS_BEFORE_UNFROZEN_ENSEMBLE_PATCH = build_model_and_loaders


def build_model_and_loaders(cfg: ExperimentConfig, data: PreparedData):
    if cfg.model.model_type != "weighted_tt_gcmc_ensemble":
        return _ORIG_BUILD_MODEL_AND_LOADERS_BEFORE_UNFROZEN_ENSEMBLE_PATCH(cfg, data)

    device = torch.device(cfg.train.device)

    warm_candidate_items = np.sort(
        data.train_df["item_idx"].astype(np.int64).unique()
    )

    seq_ds = SequencePairwiseDataset(
        data.train_df,
        warm_candidate_items,
        cfg.train.positive_threshold,
        cfg.train.max_history,
        cfg.train.min_history_for_sequence,
    )

    seq_loader = make_loader(
        seq_ds,
        cfg.train.batch_size,
        True,
        cfg,
        collate_fn=sequence_collate_fn,
    )

    tt_model = make_two_tower_model_for_hybrid(cfg, data, device)
    graph_model = make_graph_model_for_hybrid(cfg, data, device)

    load_state_dict_if_available(
        tt_model,
        cfg.model.hybrid.two_tower_checkpoint_path,
        strict=False,
        require=cfg.model.hybrid.require_pretrained,
        name="two_tower_model",
    )

    load_state_dict_if_available(
        graph_model,
        cfg.model.hybrid.graph_checkpoint_path,
        strict=False,
        require=cfg.model.hybrid.require_pretrained,
        name="graph_model",
    )

    model = WeightedTwoTowerGCMCEnsemble(
        tt_model,
        graph_model,
        cfg.model.hybrid,
    ).to(device)

    # Ключевой фикс:
    # если GCMC разморожен, НЕ строим graph-cache на этапе build.
    # Он будет строиться внутри train loop на каждый batch.
    if bool(getattr(model, "freeze_graph_encoder", True)):
        model.refresh_graph_cache()
    else:
        model._graph_user_z = None
        model._graph_item_z = None

    optimizer = torch.optim.Adam(
        (p for p in model.parameters() if p.requires_grad),
        lr=cfg.train.lr,
        weight_decay=cfg.train.weight_decay,
    )

    return model, optimizer, seq_loader


# ------------------------------------------------------------
# 3. Safe train loop for unfrozen weighted ensemble
# ------------------------------------------------------------

def train_one_epoch_pairwise_unfrozen_weighted_ensemble_safe(
    model,
    loader,
    optimizer,
    cfg: ExperimentConfig,
    logger: CometLogger,
    epoch: int,
) -> Dict[str, float]:
    device = torch.device(cfg.train.device)
    base_model = unwrap_model(model)

    model.train()

    if hasattr(base_model, "two_tower_model") and not bool(getattr(base_model, "freeze_two_tower_encoder", True)):
        base_model.two_tower_model.train()

    if hasattr(base_model, "graph_model") and not bool(getattr(base_model, "freeze_graph_encoder", True)):
        base_model.graph_model.train()

    scaler = make_grad_scaler(cfg, device)

    total_loss = 0.0
    total_steps = 0
    total_examples = 0

    step_times = []
    grad_norms = []
    graph_refresh_times = []

    n_batches = len(loader)

    _clear_graph_cache_if_present(model)

    pbar = tqdm(
        loader,
        desc=f"Train unfrozen ensemble epoch {epoch}",
        disable=not cfg.train.verbose,
    )

    for step, batch in enumerate(pbar, start=1):
        step_start = time.perf_counter()

        batch = to_device(batch, device)
        batch_size = int(batch["user_idx"].shape[0])

        optimizer.zero_grad(set_to_none=True)
        _clear_graph_cache_if_present(model)

        # Одноразовый graph-cache для текущего batch-а.
        refresh_start = time.perf_counter()
        base_model.refresh_graph_cache()
        graph_refresh_time = time.perf_counter() - refresh_start
        graph_refresh_times.append(float(graph_refresh_time))

        with get_autocast_context(cfg, device):
            pos_scores, neg_scores = base_model.score_batch(batch)
            sample_w = rating_weight(batch["rating"], cfg.train.train_rating_weighting)
            loss = bpr_loss(pos_scores, neg_scores, sample_w)

        # retain_graph НЕ нужен: cache одноразовый.
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = compute_grad_norm(
            (p for p in model.parameters() if p.requires_grad)
        )

        if cfg.train.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad),
                cfg.train.grad_clip_norm,
            )

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        # Ключевой фикс:
        # старый graph-cache после backward больше использовать нельзя.
        _clear_graph_cache_if_present(model)

        step_time = time.perf_counter() - step_start

        total_loss += float(loss.item())
        total_examples += batch_size
        total_steps += 1

        step_times.append(float(step_time))
        grad_norms.append(float(grad_norm))

        log_payload = {
            "train/loss_step": float(loss.item()),
            "train/step_time_sec": float(step_time),
            "train/grad_norm_step": float(grad_norm),
            "train/examples_per_sec": float(batch_size / max(step_time, 1e-8)),
            "train/graph_refresh_time_sec": float(graph_refresh_time),
        }

        if hasattr(base_model, "alpha_user"):
            log_payload["train/alpha_user"] = float(base_model.alpha_user().detach().cpu())

        if hasattr(base_model, "alpha_item"):
            log_payload["train/alpha_item"] = float(base_model.alpha_item().detach().cpu())

        if step % max(1, cfg.train.log_every_n_steps) == 0:
            logger.log_metrics(
                log_payload,
                step=(epoch - 1) * n_batches + step,
                epoch=epoch,
            )

        pbar.set_postfix(
            loss=f"{float(loss.item()):.4f}",
            grad=f"{float(grad_norm):.3f}",
            graph=f"{float(graph_refresh_time):.2f}s",
        )

        del batch, pos_scores, neg_scores, sample_w, loss

    _clear_graph_cache_if_present(model)

    out = {
        "train_loss": float(total_loss / max(total_steps, 1)),
        "train_steps": int(total_steps),
        "train_examples": int(total_examples),
        "train_step_time_mean_sec": float(np.mean(step_times)) if step_times else 0.0,
        "train_examples_per_sec": float(total_examples / max(np.sum(step_times), 1e-8)) if step_times else 0.0,
        "train_grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else 0.0,
        "train_grad_norm_max": float(np.max(grad_norms)) if grad_norms else 0.0,
        "hybrid_graph_refresh_time_mean_sec": float(np.mean(graph_refresh_times)) if graph_refresh_times else 0.0,
        "hybrid_graph_cache_mode": 1.0,
    }

    if hasattr(base_model, "alpha_user"):
        out["alpha_user"] = float(base_model.alpha_user().detach().cpu())

    if hasattr(base_model, "alpha_item"):
        out["alpha_item"] = float(base_model.alpha_item().detach().cpu())

    return out


# ------------------------------------------------------------
# 4. Patch dispatcher
# ------------------------------------------------------------

if "_ORIG_TRAIN_ONE_EPOCH_PAIRWISE_BEFORE_UNFROZEN_ENSEMBLE_PATCH" not in globals():
    _ORIG_TRAIN_ONE_EPOCH_PAIRWISE_BEFORE_UNFROZEN_ENSEMBLE_PATCH = train_one_epoch_pairwise


def train_one_epoch_pairwise(
    model,
    loader,
    optimizer,
    cfg: ExperimentConfig,
    logger: CometLogger,
    epoch: int,
) -> Dict[str, float]:
    base_model = unwrap_model(model)

    is_unfrozen_weighted_ensemble = (
        cfg.model.model_type == "weighted_tt_gcmc_ensemble"
        and hasattr(base_model, "graph_model")
        and not bool(getattr(base_model, "freeze_graph_encoder", True))
    )

    if is_unfrozen_weighted_ensemble:
        return train_one_epoch_pairwise_unfrozen_weighted_ensemble_safe(
            model,
            loader,
            optimizer,
            cfg,
            logger,
            epoch,
        )

    return _ORIG_TRAIN_ONE_EPOCH_PAIRWISE_BEFORE_UNFROZEN_ENSEMBLE_PATCH(
        model,
        loader,
        optimizer,
        cfg,
        logger,
        epoch,
    )


print("✅ Patch applied: unfrozen weighted TT+GCMC ensemble now uses single-use graph cache per batch.")

In [ ]:
# ============================================================
# Baseline: Two-Tower WITHOUT rating usage
# Rating is NOT used as:
#   - loss weight
#   - history token feature
#   - history token weight
#
# Positive samples are still selected by rating >= 4.0
# to avoid treating low ratings as positives.
# ============================================================

import os
import torch

DATASET_DIR = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
OUT_DIR = "/kaggle/working/two_tower_no_rating"

TWO_TOWER_NO_RATING_CKPT = f"{OUT_DIR}/best_two_tower.pt"

cfg = make_common_cfg(
    "two_tower",
    "baseline_two_tower_no_rating_binary_bpr_full_eval_run_01",
)

cfg.paths.dataset_dir = DATASET_DIR
cfg.paths.output_dir = OUT_DIR
cfg.model.model_type = "two_tower"

# ============================================================
# Features
# ============================================================

# Оставляем item features, потому что это two-tower content/history baseline.
# Если хочешь полностью ID-only two-tower, ниже можно поставить:
# cfg.features.use_item_features = False
# cfg.features.item_feature_mode = "none"

cfg.features.use_item_features = True
cfg.features.item_feature_mode = "add"      # none | add | project_only | concat_mlp | init_only
cfg.features.scaler_kind = "standard"
cfg.features.fit_scaler_on = "warm"
cfg.features.feature_dropout = 0.10
cfg.features.feature_hidden_dim = 256

# ============================================================
# Two-Tower architecture
# ============================================================

cfg.model.two_tower.embedding_dim = 128
cfg.model.two_tower.hidden_dim = 256
cfg.model.two_tower.dropout = 0.10

# User tower
cfg.model.two_tower.use_user_id_embedding = True
cfg.model.two_tower.use_history = True

# ============================================================
# Rating usage: OFF
# ============================================================

# Не добавляем rating embedding к history item tokens
cfg.model.two_tower.use_rating_in_history = False

# На всякий случай отключаем rating-based weighting внутри history
cfg.model.two_tower.rating_weight_fn = "none"

# Loss не взвешивается рейтингом:
# BPR(pos, neg) одинаковый для всех positive targets
cfg.train.train_rating_weighting = "none"

# ВАЖНО:
# это не "использование рейтинга как признака",
# это только правило, какие observed interactions считать positive.
cfg.train.positive_threshold = 4.0

# ============================================================
# Time usage
# ============================================================

# Время можно оставить: это не explicit rating signal.
# Если нужен совсем простой two-tower без rating и без времени,
# поставь use_time_in_history = False.
cfg.model.two_tower.use_time_in_history = True
cfg.model.two_tower.time_weight_fn = "exp"
cfg.model.two_tower.time_weight_gamma = 1e-8

# ============================================================
# SASRec-like history encoder
# ============================================================

cfg.model.two_tower.sasrec_num_layers = 2
cfg.model.two_tower.sasrec_num_heads = 4
cfg.model.two_tower.sasrec_ffn_dim = 256
cfg.model.two_tower.sasrec_attention_dropout = 0.10
cfg.model.two_tower.sasrec_max_seq_len = 512
cfg.model.two_tower.sasrec_use_causal_mask = True

# ============================================================
# Training
# ============================================================

cfg.train.seed = 42
cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"

cfg.train.epochs = 10
cfg.train.patience = 4

cfg.train.lr = 3e-4
cfg.train.weight_decay = 1e-5
cfg.train.grad_clip_norm = 5.0

# Для Two-Tower на 2xT4 обычно нормально 512 или 1024.
# Если качество слабое / шумное, начни с 512.
cfg.train.batch_size = 2048

cfg.train.neg_samples = 1
cfg.train.max_history = 50
cfg.train.min_history_for_sequence = 1

cfg.train.use_amp = True
cfg.train.use_multi_gpu = True
cfg.train.tf32 = True

cfg.train.num_workers = 4
cfg.train.persistent_workers = True
cfg.train.prefetch_factor = 2
cfg.train.log_every_n_steps = 100
cfg.train.verbose = True

# ============================================================
# Eval
# ============================================================

cfg.eval.eval_scenarios = (
    "warm",
    "cold_user",
    "cold_item",
    "both_cold",
)

cfg.eval.evaluate_cold_on_val = True
cfg.eval.evaluate_cold_on_test = True
cfg.eval.cold_user_use_support_all = True

cfg.eval.ks = (10, 20, 50)

# Можно логировать graded-метрики для анализа,
# но early stopping лучше делать по binary_ge_4,
# потому что сама модель обучается как binary/implicit baseline.
cfg.eval.metric_gain_types = (
    "binary_ge_4",
    "centered_3",
    "power",
)

cfg.eval.binary_metric_names = (
    "precision",
    "recall",
    "hitrate",
    "mrr",
    "map",
    "ndcg",
)

cfg.eval.graded_metric_names = ("ndcg",)

cfg.eval.positive_threshold = 4.0
cfg.eval.neutral_rating = 3.0

cfg.eval.power_beta = 0.2
cfg.eval.power_gamma = 2.0

# Основная метрика для best checkpoint
cfg.eval.main_metric_gain_type = "binary_ge_4"
cfg.eval.main_metric_name = "ndcg"
cfg.eval.main_metric_k = 10

cfg.eval.exclude_seen_items = True
cfg.eval.skip_users_with_no_relevant_targets = True
cfg.eval.user_batch_size = 256
cfg.eval.loss_batch_size = 4096
cfg.eval.compute_loss = True

# ============================================================
# Comet
# ============================================================

cfg.comet.enabled = True
cfg.comet.project_name = "explicit-movie-ranking"
cfg.comet.experiment_name = "baseline_two_tower_no_rating_binary_bpr_full_eval_run_01"

cfg.comet.tags = [
    "baseline",
    "two_tower",
    "no_rating",
    "binary_bpr",
    "rating_not_in_history",
    "rating_not_in_loss",
    "item_features",
    "sasrec_history",
    "full_eval",
    "warm",
    "cold_user",
    "cold_item",
    "both_cold",
]

# ============================================================
# Run
# ============================================================

tt_no_rating_model, tt_no_rating_data, tt_no_rating_history_df, tt_no_rating_best_val_metrics, tt_no_rating_test_metrics = run_training(cfg)

print("Saved Two-Tower no-rating checkpoint:", TWO_TOWER_NO_RATING_CKPT)
print("Best val metrics:")
print(tt_no_rating_best_val_metrics)

print("\nTest metrics:")
print(tt_no_rating_test_metrics)

In [ ]:
# ============================================================
# Baseline: Support-GCMC
# Item features: init_only
# Explicit ratings: rating-as-relation
# Time-aware propagation: ON
# Train loss: CE over rating classes + aux_bpr_weight * BPR
# ============================================================

import os
import torch

DATASET_DIR = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
OUT_DIR = "/kaggle/working/hybrid_two_tower_gcmc_time_init_only"

GCMC_TIME_CKPT = f"{OUT_DIR}/best_support_gcmc_time.pt"

cfg = make_common_cfg(
    "support_gcmc",
    "baseline_support_gcmc_init_only_time_decay_ce_bpr_full_eval_run_01",
)

cfg.paths.dataset_dir = DATASET_DIR
cfg.paths.output_dir = OUT_DIR
cfg.model.model_type = "support_gcmc"

# ============================================================
# Features: init_only
# ============================================================

cfg.features.use_item_features = True

# ВАЖНО:
# item features используются только для начальной инициализации item embedding.
# Дальше модель работает как ID/graph-based GCMC.
cfg.features.item_feature_mode = "init_only"

cfg.features.scaler_kind = "standard"
cfg.features.fit_scaler_on = "warm"
cfg.features.feature_dropout = 0.10
cfg.features.feature_hidden_dim = 256

# Параметры SVD-инициализации, если патч init_only уже добавлен
cfg.features.init_only_svd_n_iter = 7
cfg.features.init_only_random_state = 42

# ============================================================
# GCMC architecture
# ============================================================

cfg.model.graph.embedding_dim = 128
cfg.model.graph.hidden_dim = 128
cfg.model.graph.num_layers = 2
cfg.model.graph.dropout = 0.10

# Рейтинг как relation type / explicit edge signal
cfg.model.graph.rating_weight_fn = "linear_norm"

# Для top-K score используем математическое ожидание рейтинга
cfg.model.graph.relation_score_mode = "expected_rating"

# ============================================================
# Time-aware GCMC propagation
# ============================================================

# Главное отличие от обычного init_only GCMC
cfg.model.graph.relation_apply_time_decay = True

# Доступные варианты в твоей реализации:
# "none", "exp", "inverse", "sqrt_inverse"
cfg.model.graph.time_weight_fn = "exp"

# gamma для timestamp в секундах.
# 1e-8 — нормальный старт:
# примерно мягко ослабляет старые взаимодействия, но не убивает их полностью.
cfg.model.graph.time_weight_gamma = 1e-8

# ============================================================
# Auxiliary BPR
# ============================================================

cfg.model.graph.aux_bpr_weight = 0.3
cfg.model.graph.aux_bpr_positive_threshold = 4.0
cfg.model.graph.aux_bpr_neg_samples = 1

# ============================================================
# Support encoder for cold-user / both-cold
# ============================================================

cfg.model.graph.support_history_pooling = "weighted_mean"

cfg.model.graph.support_use_rating = True
cfg.model.graph.support_use_time = True

# Здесь тоже учитываем время внутри support history
cfg.model.graph.support_time_weight_fn = "exp"
cfg.model.graph.support_time_weight_gamma = 1e-8

cfg.model.graph.support_alignment_weight = 0.2
cfg.model.graph.support_alignment_loss = "mse"
cfg.model.graph.support_user_batch_size = 4096

# ============================================================
# Training
# ============================================================

cfg.train.seed = 42
cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"

cfg.train.epochs = 15
cfg.train.patience = 3

cfg.train.lr = 3e-4
cfg.train.weight_decay = 1e-5
cfg.train.grad_clip_norm = 5.0

# GCMC edge-batch
cfg.train.relation_edge_batch_size = 65536
cfg.train.graph_loss_batches_per_epoch_log = 20

cfg.train.positive_threshold = 4.0
cfg.train.train_rating_weighting = "linear_norm"

cfg.train.max_history = 50
cfg.train.min_history_for_sequence = 1

# Sparse/full-graph путь безопаснее без AMP и DataParallel
cfg.train.use_amp = False
cfg.train.use_multi_gpu = False
cfg.train.tf32 = True

cfg.train.num_workers = 4
cfg.train.persistent_workers = True
cfg.train.prefetch_factor = 2
cfg.train.log_every_n_steps = 100
cfg.train.verbose = True

# ============================================================
# Eval
# ============================================================

cfg.eval.eval_scenarios = (
    "warm",
    "cold_user",
    "cold_item",
    "both_cold",
)

cfg.eval.evaluate_cold_on_val = True
cfg.eval.evaluate_cold_on_test = True
cfg.eval.cold_user_use_support_all = True

cfg.eval.ks = (10, 20, 50)

cfg.eval.metric_gain_types = (
    "binary_ge_4",
    "centered_3",
)

cfg.eval.binary_metric_names = (
    "precision",
    "recall",
    "hitrate",
    "mrr",
    "map",
    "ndcg",
)

cfg.eval.graded_metric_names = ("ndcg",)

cfg.eval.positive_threshold = 4.0
cfg.eval.neutral_rating = 3.0

# Основная метрика для best checkpoint
cfg.eval.main_metric_gain_type = "centered_3"
cfg.eval.main_metric_name = "ndcg"
cfg.eval.main_metric_k = 10

cfg.eval.exclude_seen_items = True
cfg.eval.skip_users_with_no_relevant_targets = True
cfg.eval.user_batch_size = 128
cfg.eval.loss_batch_size = 2048
cfg.eval.compute_loss = True

# ============================================================
# Comet
# ============================================================

cfg.comet.enabled = True
cfg.comet.project_name = "explicit-movie-ranking"
cfg.comet.experiment_name = "baseline_support_gcmc_init_only_time_decay_ce_bpr_full_eval_run_01"

cfg.comet.tags = [
    "baseline",
    "support_gcmc",
    "rating_as_relation",
    "temporal",
    "time_decay",
    "init_only_features",
    "ce",
    "bpr",
    "ce_plus_bpr",
    "full_eval",
    "warm",
    "cold_user",
    "cold_item",
    "both_cold",
]

# ============================================================
# Run
# ============================================================

gcmc_time_model, gcmc_time_data, gcmc_time_history_df, gcmc_time_best_val_metrics, gcmc_time_test_metrics = run_training(cfg)

print("Saved time-aware GCMC checkpoint:", GCMC_TIME_CKPT)
print("Best warm val:")
print(gcmc_time_best_val_metrics)

print("\nTest metrics:")
print(gcmc_time_test_metrics)

In [ ]:
# ============================================================
# Baseline / Pretrain: Support-GCMC WITHOUT item features
# Train loss: CE over explicit rating classes + aux_bpr_weight * BPR
# ============================================================

import os
import torch

DATASET_DIR = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
OUT_DIR = "/kaggle/working/hybrid_two_tower_gcmc_no_features"

GCMC_CKPT_NO_FEATURES = f"{OUT_DIR}/best_support_gcmc_no_feat.pt"

cfg = make_common_cfg(
    "support_gcmc",
    "baseline_support_gcmc_no_features_ce_bpr_warm_cold_user_run_01",
)

cfg.paths.dataset_dir = DATASET_DIR
cfg.paths.output_dir = OUT_DIR
cfg.model.model_type = "support_gcmc"

# ============================================================
# Features: OFF
# ============================================================

cfg.features.use_item_features = False
cfg.features.item_feature_mode = "none"

# Можно оставить standard, но раз признаки не используются,
# scaler тоже смысла не имеет.
cfg.features.scaler_kind = "none"
cfg.features.fit_scaler_on = "warm"

cfg.features.feature_dropout = 0.0
cfg.features.feature_hidden_dim = 256

# ============================================================
# GCMC architecture
# ============================================================

cfg.model.graph.embedding_dim = 128
cfg.model.graph.hidden_dim = 128
cfg.model.graph.num_layers = 2
cfg.model.graph.dropout = 0.10

# Рейтинг используется как relation type + для весов в aux BPR
cfg.model.graph.rating_weight_fn = "linear_norm"

# Для ранжирования из GCMC лучше expected_rating:
# score = E[rating] по распределению рейтингов
cfg.model.graph.relation_score_mode = "expected_rating"

# Без time decay для чистого baseline
cfg.model.graph.relation_apply_time_decay = False
cfg.model.graph.time_weight_fn = "exp"
cfg.model.graph.time_weight_gamma = 1e-8

# Auxiliary BPR поверх CE.
# Если нужен абсолютно чистый GCMC rating-prediction baseline,
# поставь cfg.model.graph.aux_bpr_weight = 0.0
cfg.model.graph.aux_bpr_weight = 0.3
cfg.model.graph.aux_bpr_positive_threshold = 4.0
cfg.model.graph.aux_bpr_neg_samples = 1

# ============================================================
# Support encoder for cold-user
# ============================================================

cfg.model.graph.support_history_pooling = "weighted_mean"
cfg.model.graph.support_use_rating = True
cfg.model.graph.support_use_time = True
cfg.model.graph.support_time_weight_fn = "exp"
cfg.model.graph.support_time_weight_gamma = 1e-8

# Выравнивает support-представление пользователя
# с обычным user embedding на warm users
cfg.model.graph.support_alignment_weight = 0.2
cfg.model.graph.support_alignment_loss = "mse"
cfg.model.graph.support_user_batch_size = 4096

# ============================================================
# Training
# ============================================================

cfg.train.seed = 42
cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"

cfg.train.epochs = 15
cfg.train.patience = 3

cfg.train.lr = 3e-4
cfg.train.weight_decay = 1e-5
cfg.train.grad_clip_norm = 5.0

# Для graph CE training это главный batch size
cfg.train.relation_edge_batch_size = 65536
cfg.train.graph_loss_batches_per_epoch_log = 20

# Для aux BPR sample weights
cfg.train.positive_threshold = 4.0
cfg.train.train_rating_weighting = "linear_norm"

# Для support histories
cfg.train.max_history = 50
cfg.train.min_history_for_sequence = 3

# Sparse/full-graph путь: безопаснее без AMP и без DataParallel
cfg.train.use_amp = False
cfg.train.use_multi_gpu = False
cfg.train.tf32 = True

cfg.train.num_workers = 4
cfg.train.persistent_workers = True
cfg.train.prefetch_factor = 2
cfg.train.log_every_n_steps = 100
cfg.train.verbose = True

# ============================================================
# Eval
# ============================================================

# Без фичей cold_item/both_cold лучше не делать основным сравнением:
# cold items не имеют содержательного content-based представления.
cfg.eval.eval_scenarios = ("warm", "cold_user")

cfg.eval.evaluate_cold_on_val = True
cfg.eval.evaluate_cold_on_test = True
cfg.eval.cold_user_use_support_all = True

cfg.eval.ks = (10, 20, 50)

cfg.eval.metric_gain_types = ("binary_ge_4", "centered_3")
cfg.eval.binary_metric_names = (
    "precision",
    "recall",
    "hitrate",
    "mrr",
    "map",
    "ndcg",
)
cfg.eval.graded_metric_names = ("ndcg",)

cfg.eval.positive_threshold = 4.0
cfg.eval.neutral_rating = 3.0

# Основная метрика для best checkpoint
cfg.eval.main_metric_gain_type = "centered_3"
cfg.eval.main_metric_name = "ndcg"
cfg.eval.main_metric_k = 10

cfg.eval.exclude_seen_items = True
cfg.eval.skip_users_with_no_relevant_targets = True
cfg.eval.user_batch_size = 128
cfg.eval.loss_batch_size = 2048
cfg.eval.compute_loss = True

# ============================================================
# Comet
# ============================================================

cfg.comet.enabled = True
cfg.comet.project_name = "explicit-movie-ranking"
cfg.comet.experiment_name = "baseline_support_gcmc_no_features_ce_bpr_warm_cold_user_run_01"
cfg.comet.tags = [
    "baseline",
    "support_gcmc",
    "rating_as_relation",
    "no_item_features",
    "id_only_items",
    "ce",
    "bpr",
    "ce_plus_bpr",
    "warm",
    "cold_user",
]

# ============================================================
# Run
# ============================================================

gcmc_nf_model, gcmc_nf_data, gcmc_nf_history_df, gcmc_nf_best_val_metrics, gcmc_nf_test_metrics = run_training(cfg)

print("Saved GCMC no-features checkpoint:", GCMC_CKPT_NO_FEATURES)
print("Best warm val:")
print(gcmc_nf_best_val_metrics)

print("\nTest metrics:")
print(gcmc_nf_test_metrics)

In [ ]:
# ============================================================
# Weighted Ensemble: Two-Tower + Support-GCMC
# pretrained checkpoints, unfrozen joint fine-tuning
# ============================================================

import os

DATASET_DIR = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
OUT_DIR = "/kaggle/working/hybrid_two_tower_gcmc"

TT_CKPT = f"{OUT_DIR}/best_two_tower.pt"
GCMC_CKPT = f"{OUT_DIR}/best_support_gcmc.pt"

assert os.path.exists(TT_CKPT), f"Two-Tower checkpoint not found: {TT_CKPT}"
assert os.path.exists(GCMC_CKPT), f"GCMC checkpoint not found: {GCMC_CKPT}"

cfg = make_common_cfg(
    "weighted_tt_gcmc_ensemble",
    "weighted_ensemble_unfrozen_tt_gcmc_joint_finetune_run_01",
)

cfg.paths.dataset_dir = DATASET_DIR
cfg.paths.output_dir = OUT_DIR
cfg.model.model_type = "weighted_tt_gcmc_ensemble"

# ============================================================
# Features: same as in pretraining
# ============================================================

cfg.features.use_item_features = True
cfg.features.item_feature_mode = "init_only"
cfg.features.scaler_kind = "standard"
cfg.features.fit_scaler_on = "warm"
cfg.features.feature_dropout = 0.10
cfg.features.feature_hidden_dim = 256

# ============================================================
# Load pretrained branches
# ============================================================

cfg.model.hybrid.two_tower_checkpoint_path = TT_CKPT
cfg.model.hybrid.graph_checkpoint_path = GCMC_CKPT
cfg.model.hybrid.require_pretrained = True

# ============================================================
# Ensemble fusion
# ============================================================

cfg.model.hybrid.fusion_alpha_init = 0.50

# Если True, модель учит отдельно:
# alpha_user — баланс user embeddings
# alpha_item — баланс item embeddings
cfg.model.hybrid.separate_user_item_alpha = True

# Проекции полезны, если распределения embedding-ов у TT и GCMC отличаются
cfg.model.hybrid.use_projection_adapters = True

# Главное для твоей логики:
# обе предобученные модели НЕ замораживаем
cfg.model.hybrid.freeze_two_tower_encoder = True
cfg.model.hybrid.freeze_graph_encoder = True

# ============================================================
# Two-Tower architecture
# Должно совпадать с pretraining-конфигом Two-Tower
# ============================================================

cfg.model.two_tower.embedding_dim = 128
cfg.model.two_tower.hidden_dim = 256
cfg.model.two_tower.dropout = 0.10

cfg.model.two_tower.use_user_id_embedding = True
cfg.model.two_tower.use_history = True
cfg.model.two_tower.history_pooling = "weighted_mean"

cfg.model.two_tower.use_rating_in_history = True
cfg.model.two_tower.use_time_in_history = True

cfg.model.two_tower.rating_weight_fn = "none"
cfg.model.two_tower.time_weight_fn = "exp"
cfg.model.two_tower.time_weight_gamma = 1e-8

cfg.model.two_tower.sasrec_num_layers = 2
cfg.model.two_tower.sasrec_num_heads = 4
cfg.model.two_tower.sasrec_ffn_dim = 256
cfg.model.two_tower.sasrec_attention_dropout = 0.10
cfg.model.two_tower.sasrec_max_seq_len = 512
cfg.model.two_tower.sasrec_use_causal_mask = True

# ============================================================
# GCMC architecture
# Должно совпадать с pretraining-конфигом GCMC
# ============================================================

cfg.model.graph.embedding_dim = 128
cfg.model.graph.hidden_dim = 128
cfg.model.graph.num_layers = 2
cfg.model.graph.dropout = 0.10

cfg.model.graph.rating_weight_fn = "linear_norm"
cfg.model.graph.relation_score_mode = "expected_rating"
cfg.model.graph.relation_apply_time_decay = False

# Если в обновлённом ноутбуке GCMC обучался с CE + BPR,
# эти параметры лучше оставить теми же
cfg.model.graph.aux_bpr_weight = 0.3
cfg.model.graph.aux_bpr_positive_threshold = 4.0

# Support encoder для cold-user
cfg.model.graph.support_history_pooling = "weighted_mean"
cfg.model.graph.support_use_rating = True
cfg.model.graph.support_use_time = True
cfg.model.graph.support_time_weight_fn = "exp"
cfg.model.graph.support_time_weight_gamma = 1e-8

cfg.model.graph.support_alignment_weight = 0.2
cfg.model.graph.support_alignment_loss = "mse"
cfg.model.graph.support_user_batch_size = 4096

# ============================================================
# Training
# ============================================================

cfg.train.seed = 42
cfg.train.device = "cuda:0"

# При размороженном GCMC лучше начать осторожно
cfg.train.batch_size = 4096
cfg.train.neg_samples = 1

cfg.train.epochs = 10
cfg.train.patience = 3

# Для joint fine-tuning лучше маленький lr
cfg.train.lr = 1e-4
cfg.train.weight_decay = 1e-5
cfg.train.grad_clip_norm = 5.0

cfg.train.max_history = 50
cfg.train.min_history_for_sequence = 1

cfg.train.positive_threshold = 4.0
cfg.train.train_rating_weighting = "none"

# Для размороженного graph encoder безопаснее без AMP
cfg.train.use_amp = False
cfg.train.use_multi_gpu = False
cfg.train.tf32 = True

cfg.train.num_workers = 4
cfg.train.persistent_workers = True
cfg.train.prefetch_factor = 2

cfg.train.log_every_n_steps = 100
cfg.train.verbose = True

# ============================================================
# Eval
# ============================================================

cfg.eval.eval_scenarios = ("warm", "cold_user")
cfg.eval.evaluate_cold_on_val = True
cfg.eval.evaluate_cold_on_test = True
cfg.eval.cold_user_use_support_all = True

cfg.eval.ks = (10, 20, 50)

cfg.eval.metric_gain_types = ("binary_ge_4", "centered_3")
cfg.eval.binary_metric_names = (
    "precision",
    "recall",
    "hitrate",
    "mrr",
    "map",
    "ndcg",
)
cfg.eval.graded_metric_names = ("ndcg",)

cfg.eval.power_beta = 0.2
cfg.eval.power_gamma = 2.0

# Основная метрика для best checkpoint
cfg.eval.main_metric_gain_type = "centered_3"
cfg.eval.main_metric_name = "ndcg"
cfg.eval.main_metric_k = 10

cfg.eval.user_batch_size = 128
cfg.eval.loss_batch_size = 2048
cfg.eval.compute_loss = True

# ============================================================
# Comet
# ============================================================

cfg.comet.enabled = True
cfg.comet.project_name = "explicit-movie-ranking"
cfg.comet.experiment_name = "weighted_ensemble_frozen_tt_gcmc_joint_finetune_run_01"

cfg.comet.tags = [
    "hybrid",
    "weighted_ensemble",
    "two_tower",
    "support_gcmc",
    "pretrained",
    "unfrozen",
    "joint_finetune",
    "init_only_features",
    "ce_bpr_gcmc",
    "full_eval",
]

# ============================================================
# Run
# ============================================================

ensemble_model, ensemble_data, ensemble_history_df, ensemble_best_val_metrics, ensemble_test_metrics = run_training(cfg)

print("Best val metrics:")
print(ensemble_best_val_metrics)

print("\nTest metrics:")
print(ensemble_test_metrics)

# ============================================================
# Inspect learned ensemble weights
# ============================================================

base = unwrap_model(ensemble_model)

print("\nLearned fusion coefficients:")

if hasattr(base, "alpha_user"):
    print("alpha_user =", float(base.alpha_user().detach().cpu()))

if hasattr(base, "alpha_item"):
    print("alpha_item =", float(base.alpha_item().detach().cpu()))

if hasattr(base, "alpha"):
    print("alpha =", float(base.alpha().detach().cpu()))

In [ ]:
# ============================================================
# Weighted Ensemble: Two-Tower + Support-GCMC
# pretrained checkpoints, unfrozen joint fine-tuning
# ============================================================

import os

DATASET_DIR = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
OUT_DIR = "/kaggle/working/hybrid_two_tower_gcmc"

TT_CKPT = f"{OUT_DIR}/best_two_tower.pt"
GCMC_CKPT = f"{OUT_DIR}/best_support_gcmc.pt"

assert os.path.exists(TT_CKPT), f"Two-Tower checkpoint not found: {TT_CKPT}"
assert os.path.exists(GCMC_CKPT), f"GCMC checkpoint not found: {GCMC_CKPT}"

cfg = make_common_cfg(
    "weighted_tt_gcmc_ensemble",
    "weighted_ensemble_unfrozen_tt_gcmc_joint_finetune_run_01",
)

cfg.paths.dataset_dir = DATASET_DIR
cfg.paths.output_dir = OUT_DIR
cfg.model.model_type = "weighted_tt_gcmc_ensemble"

# ============================================================
# Features: same as in pretraining
# ============================================================

cfg.features.use_item_features = True
cfg.features.item_feature_mode = "init_only"
cfg.features.scaler_kind = "standard"
cfg.features.fit_scaler_on = "warm"
cfg.features.feature_dropout = 0.10
cfg.features.feature_hidden_dim = 256

# ============================================================
# Load pretrained branches
# ============================================================

cfg.model.hybrid.two_tower_checkpoint_path = TT_CKPT
cfg.model.hybrid.graph_checkpoint_path = GCMC_CKPT
cfg.model.hybrid.require_pretrained = True

# ============================================================
# Ensemble fusion
# ============================================================

cfg.model.hybrid.fusion_alpha_init = 0.50

# Если True, модель учит отдельно:
# alpha_user — баланс user embeddings
# alpha_item — баланс item embeddings
cfg.model.hybrid.separate_user_item_alpha = True

# Проекции полезны, если распределения embedding-ов у TT и GCMC отличаются
cfg.model.hybrid.use_projection_adapters = True

# Главное для твоей логики:
# обе предобученные модели НЕ замораживаем
cfg.model.hybrid.freeze_two_tower_encoder = False
cfg.model.hybrid.freeze_graph_encoder = False

# ============================================================
# Two-Tower architecture
# Должно совпадать с pretraining-конфигом Two-Tower
# ============================================================

cfg.model.two_tower.embedding_dim = 128
cfg.model.two_tower.hidden_dim = 256
cfg.model.two_tower.dropout = 0.10

cfg.model.two_tower.use_user_id_embedding = True
cfg.model.two_tower.use_history = True
cfg.model.two_tower.history_pooling = "weighted_mean"

cfg.model.two_tower.use_rating_in_history = True
cfg.model.two_tower.use_time_in_history = True

cfg.model.two_tower.rating_weight_fn = "none"
cfg.model.two_tower.time_weight_fn = "exp"
cfg.model.two_tower.time_weight_gamma = 1e-8

cfg.model.two_tower.sasrec_num_layers = 2
cfg.model.two_tower.sasrec_num_heads = 4
cfg.model.two_tower.sasrec_ffn_dim = 256
cfg.model.two_tower.sasrec_attention_dropout = 0.10
cfg.model.two_tower.sasrec_max_seq_len = 512
cfg.model.two_tower.sasrec_use_causal_mask = True

# ============================================================
# GCMC architecture
# Должно совпадать с pretraining-конфигом GCMC
# ============================================================

cfg.model.graph.embedding_dim = 128
cfg.model.graph.hidden_dim = 128
cfg.model.graph.num_layers = 2
cfg.model.graph.dropout = 0.10

cfg.model.graph.rating_weight_fn = "linear_norm"
cfg.model.graph.relation_score_mode = "expected_rating"
cfg.model.graph.relation_apply_time_decay = False

# Если в обновлённом ноутбуке GCMC обучался с CE + BPR,
# эти параметры лучше оставить теми же
cfg.model.graph.aux_bpr_weight = 0.3
cfg.model.graph.aux_bpr_positive_threshold = 4.0

# Support encoder для cold-user
cfg.model.graph.support_history_pooling = "weighted_mean"
cfg.model.graph.support_use_rating = True
cfg.model.graph.support_use_time = True
cfg.model.graph.support_time_weight_fn = "exp"
cfg.model.graph.support_time_weight_gamma = 1e-8

cfg.model.graph.support_alignment_weight = 0.2
cfg.model.graph.support_alignment_loss = "mse"
cfg.model.graph.support_user_batch_size = 4096

# ============================================================
# Training
# ============================================================

cfg.train.seed = 42
cfg.train.device = "cuda:0"

# При размороженном GCMC лучше начать осторожно
cfg.train.batch_size = 4096
cfg.train.neg_samples = 1

cfg.train.epochs = 5
cfg.train.patience = 3

# Для joint fine-tuning лучше маленький lr
cfg.train.lr = 1e-4
cfg.train.weight_decay = 1e-5
cfg.train.grad_clip_norm = 5.0

cfg.train.max_history = 50
cfg.train.min_history_for_sequence = 1

cfg.train.positive_threshold = 4.0
cfg.train.train_rating_weighting = "none"

# Для размороженного graph encoder безопаснее без AMP
cfg.train.use_amp = False
cfg.train.use_multi_gpu = False
cfg.train.tf32 = True

cfg.train.num_workers = 4
cfg.train.persistent_workers = True
cfg.train.prefetch_factor = 2

cfg.train.log_every_n_steps = 100
cfg.train.verbose = True

# ============================================================
# Eval
# ============================================================

cfg.eval.eval_scenarios = ("warm", "cold_user", "cold_item", "both_cold")
cfg.eval.evaluate_cold_on_val = True
cfg.eval.evaluate_cold_on_test = True
cfg.eval.cold_user_use_support_all = True

cfg.eval.ks = (10, 20, 50)

cfg.eval.metric_gain_types = ("binary_ge_4", "centered_3")
cfg.eval.binary_metric_names = (
    "precision",
    "recall",
    "hitrate",
    "mrr",
    "map",
    "ndcg",
)
cfg.eval.graded_metric_names = ("ndcg",)

cfg.eval.power_beta = 0.2
cfg.eval.power_gamma = 2.0

# Основная метрика для best checkpoint
cfg.eval.main_metric_gain_type = "centered_3"
cfg.eval.main_metric_name = "ndcg"
cfg.eval.main_metric_k = 10

cfg.eval.user_batch_size = 128
cfg.eval.loss_batch_size = 2048
cfg.eval.compute_loss = True

# ============================================================
# Comet
# ============================================================

cfg.comet.enabled = True
cfg.comet.project_name = "explicit-movie-ranking"
cfg.comet.experiment_name = "weighted_ensemble_unfrozen_tt_gcmc_joint_finetune_run_01"

cfg.comet.tags = [
    "hybrid",
    "weighted_ensemble",
    "two_tower",
    "support_gcmc",
    "pretrained",
    "unfrozen",
    "joint_finetune",
    "init_only_features",
    "ce_bpr_gcmc",
    "full_eval",
]

# ============================================================
# Run
# ============================================================

ensemble_model, ensemble_data, ensemble_history_df, ensemble_best_val_metrics, ensemble_test_metrics = run_training(cfg)

print("Best val metrics:")
print(ensemble_best_val_metrics)

print("\nTest metrics:")
print(ensemble_test_metrics)

# ============================================================
# Inspect learned ensemble weights
# ============================================================

base = unwrap_model(ensemble_model)

print("\nLearned fusion coefficients:")

if hasattr(base, "alpha_user"):
    print("alpha_user =", float(base.alpha_user().detach().cpu()))

if hasattr(base, "alpha_item"):
    print("alpha_item =", float(base.alpha_item().detach().cpu()))

if hasattr(base, "alpha"):
    print("alpha =", float(base.alpha().detach().cpu()))

In [ ]:
# ============================================================
# Optional 4) Deeper ensemble fine-tune
#    Unfreezes Two-Tower branch, keeps GCMC frozen to avoid graph OOM.
#    Uncomment the final run_training line if you want this extra experiment.
# ============================================================

# assert os.path.exists(TT_CKPT), f"Two-Tower checkpoint not found: {TT_CKPT}"
# assert os.path.exists(GCMC_CKPT), f"GCMC checkpoint not found: {GCMC_CKPT}"

# cfg = make_common_cfg(
#     "weighted_tt_gcmc_ensemble",
#     "weighted_ensemble_init_only_finetune_tt_frozen_gcmc_run_01",
# )
# cfg.paths.dataset_dir = DATASET_DIR
# cfg.paths.output_dir = OUT_DIR
# cfg.model.model_type = "weighted_tt_gcmc_ensemble"
# cfg = apply_init_only_features(cfg)
# cfg = apply_common_train_runtime(cfg)

# cfg.model.hybrid.two_tower_checkpoint_path = TT_CKPT
# cfg.model.hybrid.graph_checkpoint_path = GCMC_CKPT
# cfg.model.hybrid.require_pretrained = True
# cfg.model.hybrid.fusion_alpha_init = 0.50
# cfg.model.hybrid.separate_user_item_alpha = True
# cfg.model.hybrid.use_projection_adapters = True
# cfg.model.hybrid.freeze_two_tower_encoder = False
# cfg.model.hybrid.freeze_graph_encoder = True

# cfg.train.batch_size = 128
# cfg.train.epochs = 6
# cfg.train.patience = 2
# cfg.train.lr = 1e-4
# cfg.train.weight_decay = 1e-5
# cfg.train.grad_clip_norm = 5.0
# cfg.train.max_history = 50
# cfg.train.min_history_for_sequence = 1
# cfg.train.train_rating_weighting = "none"
# cfg.train.use_amp = True
# cfg.train.use_multi_gpu = False
# cfg = apply_full_eval(cfg)

# cfg.comet.enabled = True
# cfg.comet.experiment_name = "weighted_ensemble_init_only_finetune_tt_frozen_gcmc_run_01"
# cfg.comet.tags = ["hybrid", "weighted_ensemble", "fine_tune", "train_two_tower", "frozen_gcmc", "init_only_features"]

# ensemble_ft_model, ensemble_ft_data, ensemble_ft_history_df, ensemble_ft_best_val_metrics, ensemble_ft_test_metrics = run_training(cfg)
